# 00 · Persiapan Bersama (Tahap A1–A5)
## Audit data, verifikasi label, *split* terkunci, model, dan *baseline* YOLO–CNN

Notebook ini adalah tahap pertama dari tiga notebook proyek YOLO-DERMA. Hasilnya dipakai bersama oleh dua luaran:

| Notebook | Tahap | Luaran |
|---|---|---|
| **00_Persiapan** (notebook ini) | A1–A5 | *manifest* data, label terverifikasi, *split* terkunci, model, *baseline* |
| 01_Robustness_CrossDataset | B6–B10 | artikel 2025: *Robustness and Cross-Dataset Generalization of Hybrid YOLO–CNN* |
| 02_XAI_SHAP | C11–C15 | artikel 2026: *Explainable AI* berbasis SHAP untuk tingkat keparahan dermatitis atopik pada bayi |

Alur kerja notebook ini:

1. **A1 Audit dataset**: *clone* repo, unduh objek Git LFS, hitung SHA-256 dan *perceptual hash* (pHash, dHash), cek citra rusak, duplikat persis, dan *near-duplicate*.
2. **A2 Verifikasi label**: cocokkan ID kelas di berkas label dengan `data_final.yaml`, periksa konsistensi ordinal kelas keparahan secara kolorimetrik, lalu terapkan pemetaan label.
3. **A3 *Split* dan audit *data leakage***: kelompokkan citra per sumber screenshot, per sesi pengambilan, dan per *near-duplicate*; *split* bertingkat per kelompok; kunci *manifest*.
4. **A4 Model**: latih ulang YOLO (detektor lesi satu kelas) dan EfficientNet-B0 (klasifikasi keparahan ROI) pada *split* terkunci; muat bobot lama sebagai pembanding dan verifikasi *preprocessing* serta urutan kelasnya.
5. **A5 *Baseline***: satu fungsi inferensi terstandar YOLO → ROI → CNN pada *test set* yang sah.

Semua keluaran tersimpan otomatis di Google Drive: `MyDrive/YOLO_DERMA_PFR/`.

### Struktur folder di Google Drive

```
MyDrive/YOLO_DERMA_PFR/
├── 00_kode/dermakit/          modul fungsi bersama (ditulis oleh notebook ini)
├── 01_data_cache/             salinan repo (tar) agar tidak mengunduh ulang dari GitHub LFS
├── A_PERSIAPAN/
│   ├── tables/                dataset_manifest.csv, duplicate_report.csv, verified_labels.csv,
│   │                          label_mapping.json, locked_data_split.csv, leakage_audit.csv,
│   │                          model_config.json, environment.txt, baseline_predictions_*.csv, metrics.json
│   ├── figures/
│   ├── models/                bobot YOLO dan CNN hasil latih ulang
│   └── logs/
├── PFR_2025_ROBUSTNESS/       diisi notebook 01
└── PFR_2026_XAI_SHAP/         diisi notebook 02
```

### Cara menjalankan

1. *Runtime* → *Change runtime type* → **GPU (T4 atau lebih baik)**.
2. Jalankan sel dari atas ke bawah. Pelatihan YOLO dan CNN memakan waktu terlama (perkiraan 2–3 jam pada T4 dengan pengaturan bawaan; *early stopping* dapat mempersingkatnya). Bila sesi Colab terputus, jalankan ulang dari atas: data dipulihkan dari *cache* Drive, *split* dibaca dari berkas terkunci, dan pelatihan YOLO dilanjutkan dari `last.pt`.
3. Untuk uji alur singkat, set `MODE_UJI_CEPAT = True`. Keluarannya ditulis ke folder terpisah (`YOLO_DERMA_PFR_UJI_CEPAT`) sehingga tidak bercampur dengan hasil final.

### Temuan awal pada repo yang memengaruhi rancangan

Pemeriksaan awal pada repo `barata90/YOLO-DERMA` (cabang `master`) menunjukkan beberapa kondisi yang menentukan rancangan notebook ini. Semua angka di bawah dihitung ulang secara otomatis pada sel A1–A3, sehingga yang tercantum di sini hanya gambaran awal.

1. Repo hanya berisi folder `train/` (10.000 citra 512×512 dan 10.000 berkas label) serta `data_final.yaml`. Tidak ada folder *validation*, *test*, data eksternal, maupun bobot model (`model-yolo.pt`, `best_cnn_model.h5`).
2. Nama berkas berpola `aug_<indeks>_<sumber>_png.rf.<hash>.jpg`. Kita membaca pola ini sebagai varian augmentasi dari sekitar 594 screenshot sumber (umumnya 15–18 varian per sumber). *Split* acak per citra akan menempatkan varian dari sumber yang sama di *train* dan *test* (*data leakage*). Kita memakai *split* per kelompok sumber.
3. `data_final.yaml` mencatat `nc: 3` (Mild, Moderate, Severe), tetapi berkas label memuat lima ID kelas (0–4). Kita memperlakukan ini sebagai temuan verifikasi label (A2). Pemetaan bawaan `LABEL_MAP = {2: Ringan, 3: Sedang, 4: Berat}` adalah **hipotesis** yang diuji secara kolorimetrik di A2 dan wajib dikonfirmasi oleh anotator/dokter sebelum hasil dilaporkan.
4. `data_final.yaml` menunjuk `train: /content/dataset_10000/train/images`. Artinya bobot lama kemungkinan besar sudah melihat seluruh citra repo. Kita melatih ulang model pada *split* terkunci untuk evaluasi yang sah, dan bobot lama hanya dipakai sebagai pembanding berlabel *in-sample*.

## 0. Konfigurasi

Semua parameter yang boleh diubah ada di sel berikut. Nilai bawaan mengikuti rancangan analisis dan paper rujukan model *hybrid* YOLO–CNN (Setiawan et al., 2026): YOLO 640 piksel, EfficientNet-B0 224 piksel, padding ROI 10–15%, dan *weighted cross-entropy* untuk ketidakseimbangan kelas.

Catatan parameter penting:

- `LABEL_MAP` memetakan ID mentah di berkas label ke indeks keparahan (0 = Ringan, 1 = Sedang, 2 = Berat). ID yang tidak dipetakan dikeluarkan dari analisis keparahan dan dari target deteksi lesi.
- `STATUS_LABEL_MAP` ikut tercetak di setiap tabel keluaran sebagai pengingat status verifikasi.
- `SESSION_GAP_SEC` mengelompokkan screenshot yang diambil berdekatan waktu sebagai satu sesi. Screenshot beruntun berpotensi berasal dari bayi atau video yang sama, sehingga pengelompokan ini menjadi pengganti *patient_id* yang tidak tersedia.
- `CONF_THR`, `NMS_IOU`, dan `IOU_VALID` adalah ambang operasi *pipeline*: deteksi dengan *confidence* ≥ 0,25 dipakai sebagai ROI, dan ROI prediksi dianggap valid bila IoU terhadap lesi referensi ≥ 0,5.

In [ ]:
# ======================= CONFIG UTAMA =======================
NAMA_PROYEK      = "YOLO_DERMA_PFR"      # folder induk di MyDrive
REPO_URL         = "https://github.com/barata90/YOLO-DERMA.git"
REPO_BRANCH      = "master"
REPO_LOKAL       = None      # path repo yang sudah ada; None = clone otomatis
MODE_UJI_CEPAT   = False     # True = uji alur singkat (subset kecil, epoch sedikit)
SEED             = 42

# --- Pemetaan label (A2) ---
LABEL_MAP          = {2: 0, 3: 1, 4: 2}      # ID mentah -> indeks keparahan (0 Ringan, 1 Sedang, 2 Berat)
NAMA_ID_MENTAH     = {0: "Normal (hipotesis)", 1: "Tidak diketahui", 2: "Ringan", 3: "Sedang", 4: "Berat"}
STATUS_LABEL_MAP   = "HIPOTESIS_BELUM_DIVERIFIKASI_AHLI"   # ganti "TERVERIFIKASI_AHLI" setelah dikonfirmasi
BAHASA_GAMBAR      = "en"                    # "en" = Mild/Moderate/Severe, "id" = Ringan/Sedang/Berat
MIN_SISI_KOTAK_PX  = 4                       # kotak dengan sisi < 4 piksel dikeluarkan
IOU_KOTAK_GANDA    = 0.90                    # kotak kelas sama dengan IoU > 0,90 dianggap anotasi ganda

# --- Pengelompokan anti-leakage dan split (A3) ---
SESSION_GAP_SEC    = 30
HASH_HAMMING_THR   = 8         # near-duplicate bila pHash <= 8 DAN dHash <= 8 (64 bit)
SPLIT_PROPORSI     = (0.70, 0.15, 0.15)      # train, val, test (per kelompok)
FORCE_RESPLIT      = False

# --- YOLO (A4) ---
YOLO_ARCH     = "yolov8n.pt"
YOLO_IMGSZ    = 640
YOLO_EPOCHS   = 100
YOLO_PATIENCE = 20
YOLO_BATCH    = 16

# --- CNN EfficientNet-B0 (A4) ---
CNN_INPUT           = 224
ROI_PAD             = 0.125
CNN_BATCH           = 64
CNN_EPOCHS_HEAD     = 5
CNN_EPOCHS_FT       = 30
CNN_FT_LAYERS       = 60
CNN_MAX_ROI_PER_IMG = 4
CNN_MAX_ROI_VAL     = 8

# --- Inferensi pipeline (A5, B, C) ---
CONF_THR  = 0.25
NMS_IOU   = 0.7
IOU_VALID = 0.5
MAX_DET   = 300

# --- Bobot lama sebagai pembanding ---
CARI_BOBOT_LAMA        = True
BOBOT_LAMA_YOLO        = None    # contoh: "/content/drive/MyDrive/folder_model/model-yolo.pt"
BOBOT_LAMA_CNN         = None    # contoh: "/content/drive/MyDrive/folder_model/best_cnn_model.h5"
NAMA_BERKAS_BOBOT_LAMA = ("model-yolo.pt", "best_cnn_model.h5")
OLD_YOLO_KELAS_LESI    = None    # None = semua kelas YOLO lama dihitung sebagai lesi
OLD_CNN_URUTAN_KELAS   = "auto"  # "auto" atau list, misalnya [0, 1, 2]
OLD_CNN_SKALA          = "auto"  # "auto", "0-255", "0-1", "-1-1"
OLD_ROI_PAD            = 0.125

# --- Statistik ---
B_BOOT = 1000

## 1. Lingkungan kerja, Google Drive, dan folder keluaran

Sel ini memasang paket yang belum tersedia di Colab (`ultralytics`, `imagehash`), me-*mount* Google Drive, lalu membuat struktur folder. Bila `MODE_UJI_CEPAT = True`, nama folder induk diberi akhiran `_UJI_CEPAT` dan parameter pelatihan dikecilkan agar alur dapat diuji dalam beberapa menit. Di luar Colab (misalnya Jupyter lokal), folder `drive_lokal/` dipakai sebagai pengganti Drive.

In [ ]:
import os, sys, subprocess, importlib, time, json, shutil, glob, re, math, random, warnings, tarfile, datetime
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
IN_COLAB = "google.colab" in sys.modules

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs), check=True)

for mod, pkg in [("ultralytics", "ultralytics"), ("imagehash", "imagehash")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        pip_install([pkg])

# PyTorch/torchvision/Ultralytics diimpor lebih dulu daripada TensorFlow untuk menghindari konflik pustaka bersama
import torch, torchvision, ultralytics  # noqa: F401

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE, WORK = "/content/drive/MyDrive", "/content/work"
else:
    DRIVE_BASE, WORK = os.path.abspath("drive_lokal"), os.path.abspath("work_lokal")

if MODE_UJI_CEPAT:
    NAMA_PROYEK = NAMA_PROYEK + "_UJI_CEPAT"
    YOLO_EPOCHS, YOLO_PATIENCE, YOLO_IMGSZ = 2, 2, 320
    CNN_EPOCHS_HEAD, CNN_EPOCHS_FT, CNN_INPUT = 1, 1, 128
    B_BOOT = 100
N_SUMBER_UJI = 40   # hanya dipakai pada MODE_UJI_CEPAT

ROOT = os.path.join(DRIVE_BASE, NAMA_PROYEK)
P = {
    "kode": f"{ROOT}/00_kode", "data": f"{ROOT}/01_data_cache",
    "A": f"{ROOT}/A_PERSIAPAN", "A_tab": f"{ROOT}/A_PERSIAPAN/tables", "A_fig": f"{ROOT}/A_PERSIAPAN/figures",
    "A_model": f"{ROOT}/A_PERSIAPAN/models", "A_log": f"{ROOT}/A_PERSIAPAN/logs",
    "B": f"{ROOT}/PFR_2025_ROBUSTNESS", "C": f"{ROOT}/PFR_2026_XAI_SHAP",
}
for v in P.values():
    os.makedirs(v, exist_ok=True)
os.makedirs(WORK, exist_ok=True)

gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout.strip()
print("Colab           :", IN_COLAB)
print("Folder proyek   :", ROOT)
print("Folder kerja    :", WORK)
print("GPU             :", gpu if gpu else "TIDAK ADA GPU (pelatihan akan sangat lambat)")
print("Mode uji cepat  :", MODE_UJI_CEPAT)

## 2. Modul fungsi bersama `dermakit`

Fungsi yang dipakai oleh ketiga notebook dikumpulkan dalam paket kecil `dermakit` dan ditulis ke `00_kode/dermakit/` di Drive. Notebook 01 dan 02 mengimpor paket yang sama sehingga definisi crop ROI, perturbasi, metrik, dan *bootstrap* identik di semua tahap. Sel berikut menyiapkan fungsi penulis modul.

In [ ]:
DK_DIR = os.path.join(P["kode"], "dermakit")
os.makedirs(DK_DIR, exist_ok=True)

def tulis_modul(nama, isi):
    with open(os.path.join(DK_DIR, nama), "w", encoding="utf-8") as f:
        f.write(isi.lstrip("\n"))
    print("ditulis:", os.path.join(DK_DIR, nama))

### 2.1 `core.py`: konfigurasi, *seed*, log, dan laporan lingkungan

Berisi nama kelas (Ringan/Sedang/Berat dan padanan Inggrisnya), pengaturan *seed* untuk NumPy, TensorFlow, dan PyTorch, penyimpanan JSON, SHA-256 berkas, `stable_seed` (seed deterministik per citra per kondisi untuk *Gaussian noise*), log berjalan, dan `env_report` untuk `environment.txt`.

In [ ]:
tulis_modul("core.py", r'''"""dermakit.core: konfigurasi, path, seed, log, dan laporan lingkungan kerja."""
import os, sys, json, random, time, hashlib, platform, subprocess, datetime, importlib

import numpy as np

CLASS_NAMES = {
    "id": ["Ringan", "Sedang", "Berat"],
    "en": ["Mild", "Moderate", "Severe"],
}
N_CLASSES = 3


def class_names(lang="en"):
    return CLASS_NAMES.get(lang, CLASS_NAMES["en"])


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch, torchvision  # noqa: F401  (PyTorch diimpor sebelum TensorFlow)
        torch.manual_seed(seed)
    except Exception:
        pass
    try:
        import tensorflow as tf
        tf.random.set_seed(seed)
    except Exception:
        pass


def ensure_dirs(*paths):
    for p in paths:
        os.makedirs(p, exist_ok=True)


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if np.isnan(o) else float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (datetime.datetime, datetime.date)):
        return o.isoformat()
    return str(o)


def save_json(obj, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=_json_default)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def stable_seed(*parts):
    """Seed deterministik dari gabungan teks (dipakai untuk noise per citra per kondisi)."""
    return int(hashlib.sha256("|".join(map(str, parts)).encode()).hexdigest()[:8], 16)


class RunLog:
    """Log sederhana: cetak ke layar dan tambahkan ke berkas log di Drive."""

    def __init__(self, path):
        self.path = path
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    def __call__(self, msg):
        ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{ts}] {msg}"
        print(line)
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(line + "\n")


def run_cmd(cmd, check=True, tail=20, env=None):
    """Menjalankan perintah shell dan menampilkan potongan akhir output."""
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
    out = (p.stdout or "") + (p.stderr or "")
    lines = [l for l in out.strip().splitlines() if l.strip()]
    if lines:
        print("\n".join(lines[-tail:]))
    if check and p.returncode != 0:
        raise RuntimeError(f"Perintah gagal (kode {p.returncode}): {cmd}")
    return p.returncode, out


def env_report():
    """Versi paket dan perangkat keras untuk environment.txt."""
    rows = [f"tanggal_jalan: {datetime.datetime.now().isoformat(timespec='seconds')}",
            f"python: {sys.version.split()[0]}",
            f"platform: {platform.platform()}"]
    for mod in ["numpy", "pandas", "scipy", "sklearn", "cv2", "PIL", "torch", "torchvision",
                "ultralytics", "tensorflow", "keras", "shap", "imagehash", "statsmodels",
                "matplotlib", "skimage", "roboflow", "gradio", "krippendorff", "openpyxl"]:
        try:
            m = importlib.import_module(mod)
            rows.append(f"{mod}: {getattr(m, '__version__', 'terpasang')}")
        except Exception:
            rows.append(f"{mod}: tidak terpasang")
    try:
        import torch
        rows.append(f"cuda_torch: {torch.cuda.is_available()}")
        if torch.cuda.is_available():
            rows.append(f"gpu: {torch.cuda.get_device_name(0)}")
    except Exception:
        pass
    try:
        import tensorflow as tf
        rows.append(f"gpu_tensorflow: {[d.name for d in tf.config.list_physical_devices('GPU')]}")
    except Exception:
        pass
    return "\n".join(rows)


def timer():
    t0 = time.perf_counter()
    return lambda: time.perf_counter() - t0


def save_table(df, path_noext):
    """Simpan tabel sebagai parquet (bila pyarrow tersedia) atau csv.gz. Mengembalikan path berkas."""
    os.makedirs(os.path.dirname(path_noext) or ".", exist_ok=True)
    try:
        import pyarrow  # noqa: F401
        p = path_noext + ".parquet"
        df.to_parquet(p, index=False)
    except Exception:
        p = path_noext + ".csv.gz"
        df.to_csv(p, index=False, compression="gzip")
    return p


def load_table(path_noext):
    import pandas as pd
    if os.path.exists(path_noext + ".parquet"):
        return pd.read_parquet(path_noext + ".parquet")
    if os.path.exists(path_noext + ".csv.gz"):
        return pd.read_csv(path_noext + ".csv.gz")
    raise FileNotFoundError(path_noext)


def table_exists(path_noext):
    return os.path.exists(path_noext + ".parquet") or os.path.exists(path_noext + ".csv.gz")
''')

### 2.2 `data.py`: baca citra, *hash*, label YOLO, nama sumber, dan *near-duplicate*

- `read_rgb` membaca citra sebagai RGB `uint8` dan mengembalikan `None` bila berkas tidak dapat didekode.
- `image_record` menghitung SHA-256 (duplikat persis), pHash dan dHash 64 bit (*perceptual hash*, untuk citra nyaris sama), ukuran, dan status keterbacaan. Berkas *pointer* Git LFS ditandai `lfs_pointer`.
- `parse_source` memisahkan indeks augmentasi dan nama screenshot sumber dari pola `aug_<i>_<sumber>_png.rf.<hash>`. `base_source` menyatukan unduhan ganda (`X-2-` dengan `X`), dan `screenshot_time` membaca waktu pengambilan screenshot.
- `near_duplicate_pairs` mencari pasangan dengan jarak Hamming ≤ ambang secara vektor (blok 2048 baris), dan `UnionFind` menggabungkan relasi menjadi kelompok.
- `yolo_to_xyxy` mengubah kotak YOLO ternormalisasi menjadi koordinat piksel.

In [ ]:
tulis_modul("data.py", r'''"""dermakit.data: baca citra, hash, parsing label YOLO, parsing nama sumber, dan near-duplicate."""
import os, re, datetime, hashlib

import numpy as np
import cv2
from PIL import Image

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# aug_<idx>_<sumber>_<ext>.rf.<hash>  (nama berkas hasil augmentasi pada repo)
RE_AUG = re.compile(r"^aug_(\d+)_(.+?)_(png|jpg|jpeg|webp)\.rf\.([0-9a-fA-F]+)$")
# <sumber>_<ext>.rf.<hash>  (ekspor Roboflow umum)
RE_RF = re.compile(r"^(.+?)_(png|jpg|jpeg|webp)\.rf\.([0-9a-fA-F]+)$")
# Screenshot-YYYY-MM-DD-HHMMSS(-n-)
RE_SHOT = re.compile(r"Screenshot-(\d{4})-(\d{2})-(\d{2})-(\d{2})(\d{2})(\d{2})(?:-(\d+)-)?$")


def is_lfs_pointer(path):
    try:
        if os.path.getsize(path) > 400:
            return False
        with open(path, "rb") as f:
            return f.read(40).startswith(b"version https://git-lfs")
    except Exception:
        return False


def read_rgb(path):
    """Membaca citra sebagai RGB uint8. Mengembalikan None bila gagal didekode."""
    try:
        buf = np.fromfile(path, dtype=np.uint8)
        img = cv2.imdecode(buf, cv2.IMREAD_UNCHANGED)
        if img is None:
            return None
        if img.ndim == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        elif img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGB)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return np.ascontiguousarray(img)
    except Exception:
        return None


def decode_rgb_bytes(data):
    buf = np.frombuffer(data, dtype=np.uint8)
    img = cv2.imdecode(buf, cv2.IMREAD_UNCHANGED)
    if img is None:
        return None
    if img.ndim == 2:
        return cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    if img.shape[2] == 4:
        return cv2.cvtColor(img, cv2.COLOR_BGRA2RGB)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def image_record(path):
    """SHA-256, pHash, dHash (64 bit), ukuran, mode warna, dan status keterbacaan satu citra."""
    import imagehash
    rec = {"path": path, "file_bytes": os.path.getsize(path), "readable": False, "err": "",
           "sha256": "", "phash": None, "dhash": None, "width": np.nan, "height": np.nan, "mode": ""}
    try:
        with open(path, "rb") as f:
            data = f.read()
        rec["sha256"] = hashlib.sha256(data).hexdigest()
        if data.startswith(b"version https://git-lfs"):
            rec["err"] = "lfs_pointer"
            return rec
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            im.load()
            rec["width"], rec["height"], rec["mode"] = im.size[0], im.size[1], im.mode
            rgb = im.convert("RGB")
            rec["phash"] = int(str(imagehash.phash(rgb)), 16)
            rec["dhash"] = int(str(imagehash.dhash(rgb)), 16)
        rec["readable"] = True
    except Exception as e:
        rec["err"] = f"{type(e).__name__}: {str(e)[:120]}"
    return rec


def read_yolo_label(path):
    """Membaca label YOLO (cls x y w h, ternormalisasi). Mengembalikan (array n x 5, jumlah baris rusak)."""
    rows, bad = [], 0
    if not os.path.exists(path):
        return np.zeros((0, 5), dtype=np.float64), 0
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            if len(parts) != 5:
                bad += 1
                continue
            try:
                rows.append([float(v) for v in parts])
            except ValueError:
                bad += 1
    arr = np.array(rows, dtype=np.float64) if rows else np.zeros((0, 5), dtype=np.float64)
    return arr, bad


def parse_source(stem):
    """Memisahkan indeks augmentasi dan nama sumber asli dari nama berkas."""
    m = RE_AUG.match(stem)
    if m:
        return {"aug_idx": int(m.group(1)), "source_stem": m.group(2), "src_ext": m.group(3), "rf_hash": m.group(4)}
    m = RE_RF.match(stem)
    if m:
        return {"aug_idx": -1, "source_stem": m.group(1), "src_ext": m.group(2), "rf_hash": m.group(3)}
    return {"aug_idx": -1, "source_stem": stem, "src_ext": "", "rf_hash": ""}


def base_source(source_stem):
    """Menyatukan varian unduhan ganda, misalnya 'X-2-' dengan 'X'."""
    return re.sub(r"-\d+-$", "", source_stem)


def screenshot_time(source_stem):
    m = RE_SHOT.search(source_stem)
    if not m:
        return None
    y, mo, d, h, mi, s = (int(v) for v in m.groups()[:6])
    try:
        return datetime.datetime(y, mo, d, h, mi, s)
    except ValueError:
        return None


def _popcount64(x):
    x = np.asarray(x, dtype=np.uint64)
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x).astype(np.int32)
    b = x.view(np.uint8).reshape(x.shape + (8,))
    return np.unpackbits(b, axis=-1).sum(axis=-1).astype(np.int32)


def near_duplicate_pairs(hashes_a, hashes_b=None, thr=6, chunk=2048):
    """Pasangan (i, j, jarak Hamming) dengan jarak <= thr. Bila hashes_b None, dicari di dalam hashes_a (i < j)."""
    a = np.asarray([0 if h is None else h for h in hashes_a], dtype=np.uint64)
    same = hashes_b is None
    b = a if same else np.asarray([0 if h is None else h for h in hashes_b], dtype=np.uint64)
    va = np.array([h is not None for h in hashes_a])
    vb = va if same else np.array([h is not None for h in hashes_b])
    out = []
    for s in range(0, len(a), chunk):
        blk = a[s:s + chunk]
        d = _popcount64(blk[:, None] ^ b[None, :])
        ii, jj = np.nonzero(d <= thr)
        for i, j in zip(ii, jj):
            gi = s + i
            if same and j <= gi:
                continue
            if not (va[gi] and vb[j]):
                continue
            out.append((int(gi), int(j), int(d[i, j])))
    return out


class UnionFind:
    def __init__(self, items):
        self.parent = {x: x for x in items}

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[max(ra, rb)] = min(ra, rb)

    def groups(self):
        return {x: self.find(x) for x in self.parent}


def yolo_to_xyxy(xywhn, W, H):
    """Konversi kotak YOLO ternormalisasi (x, y, w, h) ke piksel (x1, y1, x2, y2)."""
    if len(xywhn) == 0:
        return np.zeros((0, 4), dtype=np.float32)
    x, y, w, h = [xywhn[:, i] for i in range(4)]
    out = np.stack([(x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H], axis=1)
    out[:, [0, 2]] = np.clip(out[:, [0, 2]], 0, W)
    out[:, [1, 3]] = np.clip(out[:, [1, 3]], 0, H)
    return out.astype(np.float32)


def xyxy_to_yolo(xyxy, W, H):
    x1, y1, x2, y2 = [xyxy[:, i] for i in range(4)]
    return np.stack([(x1 + x2) / 2 / W, (y1 + y2) / 2 / H, (x2 - x1) / W, (y2 - y1) / H], axis=1)


def list_images(folder):
    out = []
    for root, _, files in os.walk(folder):
        for fn in files:
            if fn.lower().endswith(IMG_EXT) and not fn.startswith("."):
                out.append(os.path.join(root, fn))
    return sorted(out)
''')

### 2.3 `geometry.py`: IoU, pencocokan Hungarian, dan penanda TP deteksi

- `hungarian_match` memasangkan setiap lesi referensi (GT) dengan paling banyak satu kotak prediksi sehingga total IoU maksimum (algoritma Hungarian, `scipy.optimize.linear_sum_assignment`). Hasilnya dipakai untuk E2/E4 dan analisis rentang IoU.
- `det_tp_flags` menandai *true positive* pada sepuluh ambang IoU 0,50–0,95 dengan aturan yang sama dengan Ultralytics, sehingga mAP@0.5 dan mAP@0.5:0.95 yang kita hitung dapat dicocokkan dengan `model.val()`.
- `boxes_to_tf_norm` menambah padding ROI dan mengubah koordinat ke format `tf.image.crop_and_resize`. Fungsi ini dipakai bersama oleh pelatihan dan inferensi CNN agar crop identik.

In [ ]:
tulis_modul("geometry.py", r'''"""dermakit.geometry: IoU, pencocokan Hungarian, penanda TP deteksi, dan konversi kotak ke format crop TensorFlow."""
import numpy as np
from scipy.optimize import linear_sum_assignment

IOU_THRS = np.linspace(0.5, 0.95, 10)


def iou_matrix(a, b):
    """IoU antara kotak a (n x 4) dan b (m x 4) format xyxy."""
    a = np.asarray(a, dtype=np.float64).reshape(-1, 4)
    b = np.asarray(b, dtype=np.float64).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0])
    iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2])
    iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    union = aa[:, None] + ab[None, :] - inter
    return np.where(union > 0, inter / np.maximum(union, 1e-12), 0.0)


def hungarian_match(gt, pred, min_iou=0.0):
    """Pencocokan satu-satu GT dengan prediksi yang memaksimalkan total IoU.
    Mengembalikan array best_pred (indeks prediksi per GT, -1 bila tanpa pasangan) dan best_iou."""
    n = len(gt)
    best_pred = -np.ones(n, dtype=np.int64)
    best_iou = np.zeros(n, dtype=np.float64)
    if n == 0 or len(pred) == 0:
        return best_pred, best_iou
    iou = iou_matrix(gt, pred)
    r, c = linear_sum_assignment(-iou)
    for i, j in zip(r, c):
        if iou[i, j] > min_iou:
            best_pred[i] = j
            best_iou[i] = iou[i, j]
    return best_pred, best_iou


def det_tp_flags(gt, pred, conf=None, iou_thrs=IOU_THRS):
    """Penanda TP (n_pred x n_thr) dengan pencocokan serakah berurutan confidence menurun (gaya COCO,
    sama dengan Ultralytics 8.4): setiap prediksi mengambil GT dengan IoU tertinggi yang belum diklaim."""
    n_pred = len(pred)
    thrs = np.asarray(iou_thrs, dtype=np.float64)
    flags = np.zeros((n_pred, len(thrs)), dtype=bool)
    if len(gt) == 0 or n_pred == 0:
        return flags
    order = np.argsort(-np.asarray(conf), kind="stable") if conf is not None else np.arange(n_pred)
    iou = iou_matrix(gt, np.asarray(pred)[order])
    matched = np.zeros((len(gt), len(thrs)), dtype=bool)
    sorted_flags = np.zeros_like(flags)
    cols = np.arange(len(thrs))
    for j in np.flatnonzero((iou >= thrs.min()).any(0)):
        available = np.where(matched, 0.0, iou[:, j, None])
        k = available.argmax(0)
        correct = available[k, cols] >= thrs
        sorted_flags[j] = correct
        matched[k, cols] |= correct
    flags[order] = sorted_flags
    return flags


def pad_boxes(xyxy, pad, W, H):
    """Menambah padding proporsional (pad x lebar/tinggi tiap sisi) lalu memotong ke batas citra."""
    b = np.asarray(xyxy, dtype=np.float64).reshape(-1, 4).copy()
    if len(b) == 0:
        return b
    W = np.broadcast_to(np.asarray(W, dtype=np.float64), (len(b),))
    H = np.broadcast_to(np.asarray(H, dtype=np.float64), (len(b),))
    w = b[:, 2] - b[:, 0]
    h = b[:, 3] - b[:, 1]
    b[:, 0] = np.clip(b[:, 0] - pad * w, 0, W)
    b[:, 2] = np.clip(b[:, 2] + pad * w, 0, W)
    b[:, 1] = np.clip(b[:, 1] - pad * h, 0, H)
    b[:, 3] = np.clip(b[:, 3] + pad * h, 0, H)
    return b


def boxes_to_tf_norm(xyxy, W, H, pad):
    """Kotak piksel -> [y1, x1, y2, x2] ternormalisasi untuk tf.image.crop_and_resize.
    W dan H boleh skalar atau array per kotak. Fungsi ini dipakai bersama oleh pelatihan dan inferensi agar crop identik."""
    b = pad_boxes(xyxy, pad, W, H)
    if len(b) == 0:
        return np.zeros((0, 4), dtype=np.float32)
    Wd = np.maximum(np.broadcast_to(np.asarray(W, dtype=np.float64), (len(b),)) - 1, 1)
    Hd = np.maximum(np.broadcast_to(np.asarray(H, dtype=np.float64), (len(b),)) - 1, 1)
    out = np.stack([b[:, 1] / Hd, b[:, 0] / Wd, b[:, 3] / Hd, b[:, 2] / Wd], axis=1)
    return np.clip(out, 0.0, 1.0).astype(np.float32)
''')

### 2.4 `perturb.py`: enam gangguan kualitas citra

Gangguan diterapkan pada citra utuh **sebelum** YOLO dan ukuran citra tidak berubah, sehingga koordinat ROI referensi tetap berlaku pada citra terganggu (diperlukan untuk E3).

| Jenis | Definisi | Level bawaan |
|---|---|---|
| *Brightness* | `PIL.ImageEnhance.Brightness` (perkalian intensitas) | faktor 0,6; 0,8; 1,2 |
| *Contrast* | `PIL.ImageEnhance.Contrast` (skala terhadap rerata *grayscale*) | faktor 0,7; 0,85; 1,15 |
| *Gaussian noise* | N(0, σ²) pada skala 0–255, *seed* tetap per citra | σ = 5; 15; 25 |
| *Gaussian blur* | kernel ganjil 2⌈3σ⌉+1, *border reflect* | σ = 0,5; 1,0; 2,0 |
| *Resolution* | *downsampling* `INTER_AREA` lalu *resize* kembali `INTER_LINEAR` | 75%; 50%; 25% |
| *JPEG compression* | *encode–decode* JPEG di memori | Q = 90; 70; 40 |

Rancangan jenis gangguan mengikuti kerangka *common corruptions* (Hendrycks & Dietterich, 2019). Citra terganggu tidak pernah disimpan ulang sebagai JPEG di disk, sehingga kompresi tambahan tidak mencemari kondisi selain *JPEG compression*.

In [ ]:
tulis_modul("perturb.py", r'''"""dermakit.perturb: enam jenis gangguan kualitas citra dengan level terkontrol dan seed tetap."""
import math

import numpy as np
import cv2
from PIL import Image, ImageEnhance

from .core import stable_seed

PERTURB_SPEC_DEFAULT = {
    "brightness": {"levels": [0.6, 0.8, 1.2], "param": "faktor", "param_en": "Factor", "label_en": "Brightness"},
    "contrast": {"levels": [0.7, 0.85, 1.15], "param": "faktor", "param_en": "Factor", "label_en": "Contrast"},
    "gaussian_noise": {"levels": [5, 15, 25], "param": "sigma (skala 0-255)", "param_en": "σ (0–255 scale)", "label_en": "Gaussian noise"},
    "gaussian_blur": {"levels": [0.5, 1.0, 2.0], "param": "sigma (piksel)", "param_en": "σ (pixels)", "label_en": "Gaussian blur"},
    "resolution": {"levels": [0.75, 0.5, 0.25], "param": "skala resolusi", "param_en": "Resolution scale", "label_en": "Resolution reduction"},
    "jpeg": {"levels": [90, 70, 40], "param": "kualitas JPEG (Q)", "param_en": "JPEG quality (Q)", "label_en": "JPEG compression"},
}


def list_conditions(spec=None):
    """Daftar kondisi: baseline 'original' diikuti setiap jenis x level."""
    spec = spec or PERTURB_SPEC_DEFAULT
    conds = [{"cond_id": "original", "ptype": "original", "level": 0, "value": None, "label_en": "Original"}]
    for ptype, d in spec.items():
        for i, v in enumerate(d["levels"], start=1):
            conds.append({"cond_id": f"{ptype}_L{i}", "ptype": ptype, "level": i, "value": v,
                          "label_en": d.get("label_en", ptype)})
    return conds


def blur_kernel_size(sigma):
    """Ukuran kernel ganjil yang menampung +/- 3 sigma."""
    return int(2 * math.ceil(3 * sigma) + 1)


def apply_perturbation(img, ptype, value, seed_key=""):
    """Menerapkan satu gangguan pada citra RGB uint8 utuh (sebelum YOLO). Ukuran keluaran = ukuran masukan."""
    if ptype in (None, "original") or value is None:
        return img
    if ptype == "brightness":
        return np.asarray(ImageEnhance.Brightness(Image.fromarray(img)).enhance(float(value)))
    if ptype == "contrast":
        return np.asarray(ImageEnhance.Contrast(Image.fromarray(img)).enhance(float(value)))
    if ptype == "gaussian_noise":
        rng = np.random.default_rng(stable_seed(seed_key, ptype, value))
        noisy = img.astype(np.float32) + rng.normal(0.0, float(value), img.shape).astype(np.float32)
        return np.clip(np.rint(noisy), 0, 255).astype(np.uint8)
    if ptype == "gaussian_blur":
        k = blur_kernel_size(float(value))
        return cv2.GaussianBlur(img, (k, k), sigmaX=float(value), sigmaY=float(value),
                                borderType=cv2.BORDER_REFLECT_101)
    if ptype == "resolution":
        H, W = img.shape[:2]
        w2, h2 = max(1, int(round(W * float(value)))), max(1, int(round(H * float(value))))
        small = cv2.resize(img, (w2, h2), interpolation=cv2.INTER_AREA)
        return cv2.resize(small, (W, H), interpolation=cv2.INTER_LINEAR)
    if ptype == "jpeg":
        ok, enc = cv2.imencode(".jpg", cv2.cvtColor(img, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, int(value)])
        if not ok:
            raise RuntimeError("Encode JPEG gagal")
        return cv2.cvtColor(cv2.imdecode(enc, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    raise ValueError(f"Jenis gangguan tidak dikenal: {ptype}")
''')

### 2.5 `models.py`: pembungkus YOLO dan CNN

- `get_tf` mengaktifkan *memory growth* TensorFlow sehingga GPU dapat dipakai bersama PyTorch (Ultralytics) dalam satu sesi.
- `crop_rois` memotong ROI dengan padding lalu *resize* bilinear ke ukuran masukan CNN (skala piksel 0–255).
- `YoloRunner.predict` menjalankan YOLO secara *class-agnostic* dan mengembalikan kotak `x1, y1, x2, y2, conf` pada koordinat asli. Inferensi memakai *confidence* 0,001 agar mAP dapat dihitung; ambang operasi `CONF_THR` diterapkan setelahnya.
- `CnnRunner` menyimpan skala *preprocessing* dan urutan kelas sebuah model CNN, lalu menyusun ulang probabilitas ke urutan Ringan, Sedang, Berat.
- `build_effnet_b0` membangun EfficientNet-B0 dengan graf datar (lapisan `top_activation` dapat diakses untuk Grad-CAM di notebook 02).
- `load_keras_any` memuat `.keras`/`.h5`; bila Keras 3 gagal membaca `.h5` lama, dicoba lewat `tf_keras`.

In [ ]:
tulis_modul("models.py", r'''"""dermakit.models: pembungkus YOLO (Ultralytics) dan CNN (Keras), crop ROI, serta pembangun EfficientNet-B0."""
import os, time

import numpy as np
import cv2

from .geometry import boxes_to_tf_norm

_TF = None


def get_tf():
    """Mengimpor TensorFlow dengan memory growth agar GPU bisa dipakai bersama PyTorch (Ultralytics)."""
    global _TF
    if _TF is None:
        import tensorflow as tf
        for g in tf.config.list_physical_devices("GPU"):
            try:
                tf.config.experimental.set_memory_growth(g, True)
            except Exception:
                pass
        _TF = tf
    return _TF


def crop_rois(img, xyxy, pad, size):
    """Crop ROI (dengan padding) dan resize bilinear ke size x size. Keluaran float32 skala 0-255."""
    tf = get_tf()
    if len(xyxy) == 0:
        return np.zeros((0, size, size, 3), dtype=np.float32)
    H, W = img.shape[:2]
    boxes = boxes_to_tf_norm(np.asarray(xyxy), W, H, pad)
    out = tf.image.crop_and_resize(img[None].astype(np.float32), boxes,
                                   np.zeros(len(boxes), dtype=np.int32), (size, size), method="bilinear")
    return out.numpy()


class YoloRunner:
    """Inferensi YOLO class-agnostic. Keluaran per citra: array (n, 5) = x1, y1, x2, y2, conf (piksel asli)."""

    def __init__(self, weights, imgsz=640, nms_iou=0.7, max_det=300, keep_classes=None, name="yolo", device=None):
        from ultralytics import YOLO
        self.model = YOLO(weights)
        self.weights = weights
        self.imgsz, self.nms_iou, self.max_det = imgsz, nms_iou, max_det
        self.keep_classes = keep_classes
        self.name = name
        try:
            import torch
            self.device = device if device is not None else (0 if torch.cuda.is_available() else "cpu")
            self.half = bool(torch.cuda.is_available())
        except Exception:
            self.device, self.half = "cpu", False

    @property
    def names(self):
        return getattr(self.model, "names", {})

    def predict(self, imgs_rgb, conf=0.001, batch=16):
        outs, times = [], []
        for s in range(0, len(imgs_rgb), batch):
            chunk = [cv2.cvtColor(im, cv2.COLOR_RGB2BGR) for im in imgs_rgb[s:s + batch]]
            res = self.model.predict(chunk, conf=conf, iou=self.nms_iou, imgsz=self.imgsz, max_det=self.max_det,
                                     agnostic_nms=True, verbose=False, device=self.device, half=self.half)
            for r in res:
                b = r.boxes
                xyxy = b.xyxy.cpu().numpy() if len(b) else np.zeros((0, 4))
                cf = b.conf.cpu().numpy() if len(b) else np.zeros(0)
                if self.keep_classes is not None and len(b):
                    cl = b.cls.cpu().numpy().astype(int)
                    m = np.isin(cl, list(self.keep_classes))
                    xyxy, cf = xyxy[m], cf[m]
                outs.append(np.concatenate([xyxy, cf[:, None]], axis=1).astype(np.float32))
                sp = getattr(r, "speed", {}) or {}
                times.append(float(sum(v for v in sp.values() if v is not None)))
        return outs, times


def scale_input(x, mode):
    if mode == "0-255":
        return x
    if mode == "0-1":
        return x / 255.0
    if mode == "-1-1":
        return x / 127.5 - 1.0
    raise ValueError(mode)


class CnnRunner:
    """Klasifikasi keparahan ROI. class_order[i] = indeks keparahan untuk keluaran ke-i model."""

    def __init__(self, model, scale="0-255", class_order=None, pad=0.125, name="cnn"):
        self.model = model
        shp = model.input_shape
        if isinstance(shp, list):
            shp = shp[0]
        self.input_size = int(shp[1])
        self.n_out = int(model.output_shape[-1])
        self.scale = scale
        self.class_order = list(class_order) if class_order is not None else list(range(self.n_out))
        self.pad = pad
        self.name = name

    def crops(self, img, xyxy):
        return crop_rois(img, xyxy, self.pad, self.input_size)

    def predict_raw(self, crops, batch=256):
        if len(crops) == 0:
            return np.zeros((0, self.n_out), dtype=np.float32)
        x = scale_input(np.asarray(crops, dtype=np.float32), self.scale)
        return np.asarray(self.model.predict(x, batch_size=batch, verbose=0), dtype=np.float32)

    def predict(self, crops, batch=256):
        raw = self.predict_raw(crops, batch)
        out = np.zeros((len(raw), 3), dtype=np.float32)
        for i, k in enumerate(self.class_order):
            if 0 <= k < 3:
                out[:, k] += raw[:, i]
        s = out.sum(axis=1, keepdims=True)
        return np.where(s > 0, out / np.maximum(s, 1e-12), out)

    def predict_fn(self):
        """Fungsi probabilitas untuk SHAP: menerima crop skala 0-255."""
        return lambda x: self.predict(np.asarray(x, dtype=np.float32), batch=64)


def build_effnet_b0(n_classes=3, input_size=224, dropout=0.3, weights="imagenet"):
    """EfficientNet-B0 dengan graf datar (lapisan 'top_activation' dapat diakses untuk Grad-CAM)."""
    get_tf()
    import keras
    inp = keras.Input((input_size, input_size, 3), name="roi_input")
    base = keras.applications.EfficientNetB0(include_top=False, weights=weights, input_tensor=inp)
    x = keras.layers.GlobalAveragePooling2D(name="gap")(base.output)
    x = keras.layers.Dropout(dropout, name="dropout_head")(x)
    out = keras.layers.Dense(n_classes, activation="softmax", dtype="float32", name="severity")(x)
    return keras.Model(inp, out, name="effnetb0_severity")


def load_keras_any(path):
    """Memuat model Keras (.keras/.h5). Bila Keras 3 gagal membaca .h5 lama, dicoba lewat tf_keras."""
    get_tf()
    import keras
    try:
        return keras.models.load_model(path, compile=False, safe_mode=False), "keras"
    except Exception as e1:
        try:
            import tf_keras
            return tf_keras.models.load_model(path, compile=False), "tf_keras"
        except Exception as e2:
            raise RuntimeError(f"Gagal memuat {path}. keras: {e1}. tf_keras: {e2}")


def find_files(root, names, max_depth=5, timeout=180):
    """Mencari berkas dengan nama tertentu di bawah root (dibatasi kedalaman dan waktu)."""
    found = {n: [] for n in names}
    t0 = time.time()
    root = os.path.abspath(root)
    base_depth = root.rstrip(os.sep).count(os.sep)
    for cur, dirs, files in os.walk(root):
        if time.time() - t0 > timeout:
            break
        if cur.count(os.sep) - base_depth >= max_depth:
            dirs[:] = []
        dirs[:] = [d for d in dirs if not d.startswith(".")]
        for n in names:
            if n in files:
                found[n].append(os.path.join(cur, n))
    return found
''')

### 2.6 `pipeline.py`: satu fungsi inferensi terstandar

`run_condition` adalah satu-satunya jalur inferensi yang dipakai untuk *baseline*, perturbasi, data eksternal, dan propagasi kesalahan. Untuk setiap citra fungsi ini:

1. menerapkan gangguan (atau tidak, pada kondisi *original*);
2. menjalankan YOLO dan menyimpan penanda TP untuk mAP;
3. memasangkan lesi referensi dengan kotak prediksi (Hungarian) pada ambang `CONF_THR`;
4. menjalankan CNN pada ROI referensi (E1/E3) dan pada ROI prediksi (E2/E4);
5. mencatat `pipe_pred = -1` untuk lesi tanpa ROI valid (IoU < `IOU_VALID` atau tidak terdeteksi). Kasus ini **tetap berada di penyebut** metrik *end-to-end*.

Label tingkat citra memakai aturan keparahan maksimum: label referensi = kelas tertinggi di antara lesi GT, prediksi = kelas tertinggi di antara ROI hasil YOLO.

In [ ]:
tulis_modul("pipeline.py", r'''"""dermakit.pipeline: satu fungsi inferensi terstandar YOLO -> ROI -> CNN untuk satu kondisi citra."""
import time

import numpy as np
import pandas as pd

from .data import read_rgb
from .perturb import apply_perturbation
from .geometry import det_tp_flags, hungarian_match, IOU_THRS


def run_condition(items, yolo, cnn, cond, conf_thr=0.25, iou_valid=0.5, batch_imgs=8, cnn_batch=256,
                  progress=True, desc=""):
    """Menjalankan pipeline pada daftar citra untuk satu kondisi.

    items: list dict berisi sample_id, group, path, gt_xyxy (n x 4 piksel), gt_cls (n,).
    Keluaran:
      roi_df : satu baris per lesi GT. E-ref (ROI referensi) dan E-pred (ROI prediksi YOLO hasil Hungarian).
      img_df : satu baris per citra (jumlah deteksi, label tingkat citra, latensi).
      det    : dict sample_id -> (flags TP n_pred x 10, conf n_pred, n_gt) untuk mAP.
    Kasus tanpa deteksi tetap berada di penyebut (pipe_pred = -1).
    """
    roi_rows, img_rows, det = [], [], {}
    it_range = range(0, len(items), batch_imgs)
    if progress:
        try:
            from tqdm.auto import tqdm
            it_range = tqdm(it_range, desc=desc or cond["cond_id"], leave=False)
        except Exception:
            pass
    for s in it_range:
        chunk = items[s:s + batch_imgs]
        imgs, okm = [], []
        for it in chunk:
            im = read_rgb(it["path"])
            if im is None:
                okm.append(False)
                imgs.append(np.zeros((32, 32, 3), np.uint8))
                continue
            okm.append(True)
            imgs.append(apply_perturbation(im, cond["ptype"], cond["value"], seed_key=it["sample_id"]))
        preds, ytimes = yolo.predict(imgs, conf=0.001)
        crops, spans, pthr_list = [], [], []
        for it, im, p, ok in zip(chunk, imgs, preds, okm):
            if not ok:
                spans.append(None)
                pthr_list.append(None)
                continue
            pthr = p[p[:, 4] >= conf_thr]
            cg = cnn.crops(im, it["gt_xyxy"])
            cp = cnn.crops(im, pthr[:, :4])
            spans.append((len(cg), len(cp)))
            crops.append(cg)
            crops.append(cp)
            pthr_list.append(pthr)
        t0 = time.perf_counter()
        allc = np.concatenate(crops) if crops else np.zeros((0, cnn.input_size, cnn.input_size, 3), np.float32)
        probs = cnn.predict(allc, batch=cnn_batch)
        t_cnn = (time.perf_counter() - t0) * 1000.0
        n_total = max(len(allc), 1)
        off = 0
        for it, im, p, ok, sp, pthr, yt in zip(chunk, imgs, preds, okm, spans, pthr_list, ytimes):
            sid = it["sample_id"]
            if not ok:
                img_rows.append({"sample_id": sid, "group": it["group"], "read_ok": False})
                continue
            ng, npd = sp
            pg, pp = probs[off:off + ng], probs[off + ng:off + ng + npd]
            off += ng + npd
            gt, gcls = it["gt_xyxy"], np.asarray(it["gt_cls"], dtype=int)
            det[sid] = (det_tp_flags(gt, p[:, :4], p[:, 4], IOU_THRS), p[:, 4].copy(), len(gt))
            bp, biou = hungarian_match(gt, pthr[:, :4], min_iou=0.0)
            ref_pred = pg.argmax(1) if len(pg) else np.zeros(0, int)
            for i in range(len(gt)):
                j = int(bp[i])
                has = j >= 0
                valid = bool(has and biou[i] >= iou_valid)
                any_pred = int(pp[j].argmax()) if has else -1
                roi_rows.append({
                    "sample_id": sid, "group": it["group"], "roi_idx": i, "true": int(gcls[i]),
                    "ref_p0": pg[i, 0], "ref_p1": pg[i, 1], "ref_p2": pg[i, 2], "ref_pred": int(ref_pred[i]),
                    "iou": float(biou[i]) if has else 0.0, "has_overlap": has, "valid_roi": valid,
                    "det_conf": float(pthr[j, 4]) if has else np.nan,
                    "any_p0": pp[j, 0] if has else np.nan, "any_p1": pp[j, 1] if has else np.nan,
                    "any_p2": pp[j, 2] if has else np.nan, "any_pred": any_pred,
                    "pipe_pred": any_pred if valid else -1,
                    "pb_x1": float(pthr[j, 0]) if has else np.nan, "pb_y1": float(pthr[j, 1]) if has else np.nan,
                    "pb_x2": float(pthr[j, 2]) if has else np.nan, "pb_y2": float(pthr[j, 3]) if has else np.nan,
                    "gt_x1": float(gt[i, 0]), "gt_y1": float(gt[i, 1]), "gt_x2": float(gt[i, 2]), "gt_y2": float(gt[i, 3]),
                    "box_area_px": float((gt[i, 2] - gt[i, 0]) * (gt[i, 3] - gt[i, 1])),
                })
            img_rows.append({
                "sample_id": sid, "group": it["group"], "read_ok": True, "n_gt": len(gt),
                "n_pred": len(pthr), "any_det": len(pthr) > 0,
                "true_img": int(gcls.max()) if len(gcls) else -1,
                "ref_img": int(ref_pred.max()) if len(ref_pred) else -1,
                "pipe_img": int(pp.argmax(1).max()) if len(pp) else -1,
                "max_det_conf": float(pthr[:, 4].max()) if len(pthr) else np.nan,
                "yolo_ms": yt, "cnn_ms": t_cnn * (ng + npd) / n_total,
                "lesion_recall_at_thr": float(np.mean((bp >= 0) & (biou >= iou_valid))) if len(gt) else np.nan,
            })
    roi_df = pd.DataFrame(roi_rows)
    img_df = pd.DataFrame(img_rows)
    for df in (roi_df, img_df):
        if len(df):
            df.insert(0, "cond_id", cond["cond_id"])
    return roi_df, img_df, det


def build_items(split_df, ver_df, repo_dir, split=None):
    """Menyusun daftar citra + kotak GT terverifikasi (hanya kotak keep=True dan sev >= 0)."""
    import os
    df = split_df if split is None else split_df[split_df["split"] == split]
    v = ver_df[(ver_df["keep"].astype(bool)) & (ver_df["sev"] >= 0)]
    gb = {k: g.sort_values("box_idx") for k, g in v.groupby("sample_id")}
    items = []
    for r in df.itertuples(index=False):
        g = gb.get(r.sample_id)
        if g is None:
            xyxy, cls = np.zeros((0, 4), np.float32), np.zeros(0, int)
        else:
            xyxy = g[["x1", "y1", "x2", "y2"]].to_numpy(np.float32)
            cls = g["sev"].to_numpy(int)
        items.append({"sample_id": r.sample_id, "group": str(r.group_id),
                      "path": os.path.join(repo_dir, r.file_rel), "gt_xyxy": xyxy, "gt_cls": cls})
    return items


def save_det(det, path):
    sids = list(det.keys())
    flags = [det[s][0] for s in sids]
    confs = [det[s][1] for s in sids]
    ngt = np.array([det[s][2] for s in sids], dtype=np.int64)
    lens = np.array([len(c) for c in confs], dtype=np.int64)
    np.savez_compressed(path, sids=np.array(sids), ngt=ngt, lens=lens,
                        flags=np.concatenate(flags) if flags else np.zeros((0, 10), bool),
                        confs=np.concatenate(confs) if confs else np.zeros(0, np.float32))


def load_det(path):
    z = np.load(path, allow_pickle=False)
    sids, ngt, lens = z["sids"], z["ngt"], z["lens"]
    flags, confs = z["flags"], z["confs"]
    out, off = {}, 0
    for s, n, l in zip(sids, ngt, lens):
        out[str(s)] = (flags[off:off + l], confs[off:off + l], int(n))
        off += l
    return out
''')

### 2.7 `metrics.py`: metrik klasifikasi, kalibrasi, dan AP

- `cls_metrics`: *accuracy*, *macro-F1*, *balanced accuracy*, *precision/recall/F1* per kelas. Prediksi −1 (tanpa ROI valid) dihitung sebagai *false negative* kelas sebenarnya.
- `ece_top`: *Expected Calibration Error* top-label dengan 15 bin (Guo et al., 2017). `classwise_ece`: ECE per kelas (Kull et al., 2019).
- `brier_multiclass`: *Brier score* multikelas (Brier, 1950), rentang 0–2.
- `ap_from_det`: AP dengan interpolasi 101 titik (konvensi COCO, sama dengan Ultralytics) untuk mAP@0.5 dan mAP@0.5:0.95 detektor lesi satu kelas.

In [ ]:
tulis_modul("metrics.py", r'''"""dermakit.metrics: metrik klasifikasi (kegagalan deteksi dihitung salah), kalibrasi, dan AP deteksi."""
import numpy as np


def cls_metrics(y_true, y_pred, n=3):
    """Accuracy, macro-F1, balanced accuracy, dan metrik per kelas.
    y_pred = -1 berarti kegagalan (tanpa ROI valid): dihitung sebagai FN kelas sebenarnya, bukan FP kelas mana pun.
    Macro-F1 dan balanced accuracy dirata-ratakan atas kelas yang memiliki support > 0."""
    yt = np.asarray(y_true, dtype=int)
    yp = np.asarray(y_pred, dtype=int)
    out = {"n": int(len(yt))}
    if len(yt) == 0:
        out.update({"accuracy": np.nan, "macro_f1": np.nan, "balanced_accuracy": np.nan, "failure_rate": np.nan})
        return out
    support = np.bincount(yt, minlength=n)[:n]
    tp = np.array([np.sum((yt == k) & (yp == k)) for k in range(n)])
    pc = np.array([np.sum(yp == k) for k in range(n)])
    prec = np.where(pc > 0, tp / np.maximum(pc, 1), np.nan)
    rec = np.where(support > 0, tp / np.maximum(support, 1), np.nan)
    with np.errstate(invalid="ignore", divide="ignore"):
        f1 = np.where((support > 0), np.where(np.nan_to_num(prec) + np.nan_to_num(rec) > 0,
                      2 * np.nan_to_num(prec) * np.nan_to_num(rec) / (np.nan_to_num(prec) + np.nan_to_num(rec)), 0.0), np.nan)
    present = support > 0
    out.update({
        "accuracy": float(np.mean(yt == yp)),
        "macro_f1": float(np.nanmean(f1[present])) if present.any() else np.nan,
        "balanced_accuracy": float(np.nanmean(rec[present])) if present.any() else np.nan,
        "failure_rate": float(np.mean(yp == -1)),
    })
    for k in range(n):
        out[f"precision_{k}"] = float(prec[k]) if not np.isnan(prec[k]) else np.nan
        out[f"recall_{k}"] = float(rec[k]) if not np.isnan(rec[k]) else np.nan
        out[f"f1_{k}"] = float(f1[k]) if not np.isnan(f1[k]) else np.nan
        out[f"support_{k}"] = int(support[k])
    return out


def macro_f1_fast(yt, yp, n=3):
    yt = np.asarray(yt, dtype=np.int64)
    yp = np.asarray(yp, dtype=np.int64)
    if len(yt) == 0:
        return np.nan
    m = yp >= 0
    cm = np.bincount(yt[m] * n + yp[m], minlength=n * n).reshape(n, n)
    support = np.bincount(yt, minlength=n)[:n]
    tp = np.diag(cm)
    pc = cm.sum(0)
    prec = np.divide(tp, pc, out=np.zeros(n), where=pc > 0)
    rec = np.divide(tp, support, out=np.zeros(n), where=support > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(n), where=(prec + rec) > 0)
    pres = support > 0
    return float(f1[pres].mean()) if pres.any() else np.nan


def balanced_acc_fast(yt, yp, n=3):
    yt = np.asarray(yt, dtype=np.int64)
    yp = np.asarray(yp, dtype=np.int64)
    if len(yt) == 0:
        return np.nan
    support = np.bincount(yt, minlength=n)[:n]
    tp = np.bincount(yt[yt == yp], minlength=n)[:n]
    pres = support > 0
    return float((tp[pres] / support[pres]).mean()) if pres.any() else np.nan


def confusion(y_true, y_pred, n=3):
    """Matriks n x (n+1); kolom terakhir = tanpa ROI valid."""
    cm = np.zeros((n, n + 1), dtype=int)
    for t, p in zip(np.asarray(y_true, int), np.asarray(y_pred, int)):
        cm[t, p if p >= 0 else n] += 1
    return cm


def ece_top(probs, y, n_bins=15):
    """Expected Calibration Error top-label (Guo et al., 2017), bin lebar sama."""
    probs = np.asarray(probs, dtype=np.float64)
    y = np.asarray(y, dtype=int)
    if len(y) == 0:
        return np.nan
    conf = probs.max(1)
    acc = (probs.argmax(1) == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(conf, edges[1:-1], right=True), 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            ece += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(ece)


def classwise_ece(probs, y, n_bins=15):
    """Classwise-ECE (Kull et al., 2019): rata-rata ECE satu-lawan-lainnya per kelas."""
    probs = np.asarray(probs, dtype=np.float64)
    y = np.asarray(y, dtype=int)
    if len(y) == 0:
        return np.nan
    edges = np.linspace(0, 1, n_bins + 1)
    vals = []
    for k in range(probs.shape[1]):
        p = probs[:, k]
        t = (y == k).astype(float)
        idx = np.clip(np.digitize(p, edges[1:-1], right=True), 0, n_bins - 1)
        e = 0.0
        for b in range(n_bins):
            m = idx == b
            if m.any():
                e += m.mean() * abs(t[m].mean() - p[m].mean())
        vals.append(e)
    return float(np.mean(vals))


def brier_multiclass(probs, y):
    """Brier score multikelas: rerata jumlah kuadrat selisih probabilitas dan one-hot (rentang 0-2)."""
    probs = np.asarray(probs, dtype=np.float64)
    y = np.asarray(y, dtype=int)
    if len(y) == 0:
        return np.nan
    oh = np.zeros_like(probs)
    oh[np.arange(len(y)), y] = 1.0
    return float(np.mean(np.sum((probs - oh) ** 2, axis=1)))


def reliability_table(probs, y, n_bins=15):
    probs = np.asarray(probs, dtype=np.float64)
    y = np.asarray(y, dtype=int)
    conf = probs.max(1)
    acc = (probs.argmax(1) == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(conf, edges[1:-1], right=True), 0, n_bins - 1)
    rows = []
    for b in range(n_bins):
        m = idx == b
        rows.append({"bin": b, "lo": edges[b], "hi": edges[b + 1], "n": int(m.sum()),
                     "conf_mean": float(conf[m].mean()) if m.any() else np.nan,
                     "acc_mean": float(acc[m].mean()) if m.any() else np.nan})
    return rows


def compute_ap(recall, precision):
    """AP dengan interpolasi 101 titik (konvensi COCO, sama dengan Ultralytics 8.4):
    precision bernilai 0 pada recall di atas recall maksimum yang dicapai."""
    recall = np.asarray(recall, dtype=np.float64)
    precision = np.asarray(precision, dtype=np.float64)
    last = recall[-1] if len(recall) else 1.0
    mrec = np.concatenate(([0.0], recall, [last], [1.0]))
    mpre = np.concatenate(([1.0], precision, [0.0], [0.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    x = np.linspace(0, 1, 101)
    trap = getattr(np, "trapezoid", None) or np.trapz
    return float(trap(np.interp(x, mrec, mpre), x))


def ap_from_det(det, sample_ids=None, conf_thr=0.25):
    """mAP@0.5, mAP@0.5:0.95 (satu kelas 'lesi'), serta precision/recall pada ambang operasi."""
    keys = list(det.keys()) if sample_ids is None else list(sample_ids)
    if not keys:
        return {"map50": np.nan, "map50_95": np.nan, "precision_thr": np.nan, "recall_thr": np.nan, "n_gt": 0}
    flags = np.concatenate([det[k][0] for k in keys])
    confs = np.concatenate([det[k][1] for k in keys])
    n_gt = int(sum(det[k][2] for k in keys))
    if n_gt == 0:
        return {"map50": np.nan, "map50_95": np.nan, "precision_thr": np.nan, "recall_thr": np.nan, "n_gt": 0}
    order = np.argsort(-confs, kind="stable")
    f = flags[order]
    c = confs[order]
    tpc = np.cumsum(f, axis=0)
    fpc = np.cumsum(~f, axis=0)
    rec = tpc / n_gt
    prec = tpc / np.maximum(tpc + fpc, 1)
    aps = np.array([compute_ap(rec[:, t], prec[:, t]) for t in range(f.shape[1])]) if len(f) else np.zeros(f.shape[1])
    m = c >= conf_thr
    tp_thr = int(f[m, 0].sum())
    return {"map50": float(aps[0]), "map50_95": float(aps.mean()),
            "precision_thr": tp_thr / max(int(m.sum()), 1) if m.sum() else np.nan,
            "recall_thr": tp_thr / n_gt, "n_gt": n_gt, "n_pred_thr": int(m.sum())}
''')

### 2.8 `stats.py`: *bootstrap* klaster berpasangan

Varian augmentasi dari satu sumber tidak independen satu sama lain. Interval kepercayaan 95% dihitung dengan *cluster bootstrap* (Field & Welsh, 2007): unit yang di-*resample* adalah kelompok sumber/sesi. Satu himpunan kelompok terpilih dipakai bersama untuk semua kondisi pada replikasi yang sama, sehingga selisih antar-kondisi (ΔL, ΔC, ΔP, ΔLP) bersifat berpasangan. Nilai p bootstrap dua sisi dan koreksi Holm (1979) disediakan untuk perbandingan berganda.

In [ ]:
tulis_modul("stats.py", r'''"""dermakit.stats: bootstrap klaster berpasangan, interval persentil, nilai p bootstrap, dan koreksi Holm."""
import numpy as np


class ClusterBootstrap:
    """Resampling klaster (kelompok sumber/sesi) dengan pengembalian.
    Satu himpunan klaster terpilih dipakai bersama untuk semua tabel yang didaftarkan sehingga
    perbandingan antar-kondisi bersifat berpasangan."""

    def __init__(self, B=1000, seed=42):
        self.B, self.seed = int(B), int(seed)
        self.tables = {}
        self.universe = None

    def register(self, key, groups):
        groups = np.asarray(groups).astype(str)
        order = np.argsort(groups, kind="stable")
        g_sorted = groups[order]
        uniq, start = np.unique(g_sorted, return_index=True)
        ends = np.append(start[1:], len(g_sorted))
        self.tables[key] = {u: order[s:e] for u, s, e in zip(uniq, start, ends)}
        allg = set().union(*[set(t.keys()) for t in self.tables.values()])
        self.universe = np.array(sorted(allg))
        return self

    def draws(self):
        rng = np.random.default_rng(self.seed)
        n = len(self.universe)
        for _ in range(self.B):
            gids = self.universe[rng.integers(0, n, n)]
            out = {}
            for key, mp in self.tables.items():
                parts = [mp[g] for g in gids if g in mp]
                out[key] = np.concatenate(parts) if parts else np.zeros(0, dtype=int)
            yield out


def percentile_ci(values, alpha=0.05):
    v = np.asarray(values, dtype=float)
    v = v[~np.isnan(v)]
    if len(v) == 0:
        return (np.nan, np.nan)
    return (float(np.quantile(v, alpha / 2)), float(np.quantile(v, 1 - alpha / 2)))


def boot_p_two_sided(deltas):
    """Nilai p dua sisi bootstrap untuk H0: delta = 0 (proporsi replikasi di sisi berlawanan, dikali 2)."""
    d = np.asarray(deltas, dtype=float)
    d = d[~np.isnan(d)]
    if len(d) == 0:
        return np.nan
    p = 2 * min(np.mean(d <= 0), np.mean(d >= 0))
    return float(min(1.0, max(p, 1.0 / len(d))))


def holm(pvals):
    """Koreksi Holm (1979) untuk perbandingan berganda."""
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    idx = np.where(ok)[0]
    if len(idx) == 0:
        return out
    order = idx[np.argsort(p[idx])]
    m = len(order)
    running = 0.0
    for r, i in enumerate(order):
        running = max(running, min(1.0, (m - r) * p[i]))
        out[i] = running
    return out
''')

### 2.9 `viz.py` dan `teks.py`: gambar dan format angka

`setup_style` mengatur ukuran huruf minimum 12 pt dan *constrained layout* agar label tidak tumpang tindih. `save_fig` menyimpan PNG 300 dpi tanpa metadata *Software*, dan PDF tanpa *Creator/Producer/CreationDate*. `teks.py` memformat angka dengan koma desimal untuk interpretasi otomatis dan mengikat kalimat arah perubahan pada interval kepercayaan.

In [ ]:
tulis_modul("viz.py", r'''"""dermakit.viz: gaya gambar siap naskah (font >= 12, tanpa tumpang tindih) dan penyimpanan tanpa metadata perangkat lunak."""
import os

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

CLASS_COLORS = ["#4C9F70", "#E0A030", "#C8473D"]  # Ringan, Sedang, Berat
SET_COLORS = {"ref": "#2F6690", "pred": "#D1495B", "internal": "#2F6690", "external": "#D1495B"}
PTYPE_ORDER = ["brightness", "contrast", "gaussian_noise", "gaussian_blur", "resolution", "jpeg"]


def setup_style(base=13):
    matplotlib.rcParams.update({
        "font.size": base, "axes.titlesize": base + 1, "axes.labelsize": base,
        "xtick.labelsize": base - 1, "ytick.labelsize": base - 1, "legend.fontsize": base - 1,
        "figure.titlesize": base + 2, "figure.dpi": 110, "savefig.dpi": 300,
        "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
        "figure.constrained_layout.use": True, "font.family": "DejaVu Sans",
    })


def save_fig(fig, path_noext, formats=("png",)):
    """Simpan gambar 300 dpi tanpa metadata 'Software'/'Creator'/'Producer'/tanggal."""
    os.makedirs(os.path.dirname(path_noext) or ".", exist_ok=True)
    paths = []
    for fmt in formats:
        p = f"{path_noext}.{fmt}"
        if fmt == "png":
            meta = {"Software": None}
        elif fmt == "pdf":
            meta = {"Creator": None, "Producer": None, "CreationDate": None}
        else:
            meta = None
        fig.savefig(p, dpi=300, bbox_inches="tight", metadata=meta)
        paths.append(p)
    return paths


def show_and_save(fig, path_noext, formats=("png",)):
    paths = save_fig(fig, path_noext, formats)
    plt.show()
    plt.close(fig)
    return paths


def overlay_heatmap(img_uint8, heat, alpha=0.5, cmap="bwr", vmax=None):
    """Menumpangkan peta atribusi bertanda (merah = menaikkan, biru = menurunkan probabilitas kelas)."""
    vmax = vmax or (np.nanpercentile(np.abs(heat), 99.5) + 1e-12)
    cm = plt.get_cmap(cmap)
    rgba = cm(np.clip(heat / (2 * vmax) + 0.5, 0, 1))
    base = img_uint8.astype(np.float32) / 255.0
    gray = np.repeat(base.mean(axis=2, keepdims=True), 3, axis=2)
    out = (1 - alpha) * gray + alpha * rgba[..., :3]
    return np.clip(out, 0, 1)
''')

In [ ]:
tulis_modul("teks.py", r'''"""dermakit.teks: format angka gaya Indonesia (koma desimal) untuk interpretasi otomatis."""
import numpy as np


def fmt(x, d=3):
    try:
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return "NA"
        return f"{float(x):.{d}f}".replace(".", ",")
    except Exception:
        return str(x)


def pct(x, d=1):
    try:
        if x is None or np.isnan(x):
            return "NA"
        return f"{100 * float(x):.{d}f}%".replace(".", ",")
    except Exception:
        return str(x)


def fmt_ci(lo, hi, d=3):
    return f"[{fmt(lo, d)}; {fmt(hi, d)}]"


def ci_excludes_zero(lo, hi):
    return (not np.isnan(lo)) and (not np.isnan(hi)) and (lo > 0 or hi < 0)


def arah(delta, lo, hi, naik="meningkat", turun="menurun"):
    """Kalimat arah perubahan yang terikat pada interval kepercayaan."""
    if np.isnan(delta):
        return "tidak dapat dihitung"
    if ci_excludes_zero(lo, hi):
        return f"{naik if delta > 0 else turun} (95% CI {fmt_ci(lo, hi)} tidak mencakup nol)"
    return f"berubah {fmt(delta)} tetapi 95% CI {fmt_ci(lo, hi)} mencakup nol, sehingga arah perubahan belum dapat dipastikan"
''')

In [ ]:
tulis_modul("__init__.py", r'''"""dermakit: utilitas bersama proyek YOLO-DERMA (robustness, cross-dataset, dan XAI SHAP)."""
from . import core, data, geometry, perturb, models, pipeline, metrics, stats, viz, teks  # noqa: F401

__version__ = "1.0.0"
''')

### 2.10 Impor modul, simpan konfigurasi, dan catat lingkungan

Konfigurasi run disimpan ke `A_PERSIAPAN/config_run.json`. Notebook 01 dan 02 membaca berkas ini sehingga label map, ambang, ukuran masukan, dan *seed* tidak berbeda antartahap. Versi paket dan GPU dicatat di `environment.txt`.

In [ ]:
if P["kode"] not in sys.path:
    sys.path.insert(0, P["kode"])
import importlib
import dermakit as dk
for m in [dk.core, dk.data, dk.geometry, dk.perturb, dk.models, dk.pipeline, dk.metrics, dk.stats, dk.viz, dk.teks]:
    importlib.reload(m)
importlib.reload(dk)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
dk.core.set_seed(SEED)
dk.viz.setup_style()
LOG = dk.core.RunLog(os.path.join(P["A_log"], "run_log_00.txt"))
NAMA_KELAS = dk.core.class_names(BAHASA_GAMBAR)
NAMA_KELAS_ID = dk.core.class_names("id")

CONFIG_RUN = {k: v for k, v in dict(globals()).items()
              if k.isupper() and isinstance(v, (int, float, str, bool, tuple, list, dict, type(None)))}
CONFIG_RUN["LABEL_MAP"] = {str(k): v for k, v in LABEL_MAP.items()}
CONFIG_RUN["NAMA_ID_MENTAH"] = {str(k): v for k, v in NAMA_ID_MENTAH.items()}
CONFIG_RUN["dibuat"] = datetime.datetime.now().isoformat(timespec="seconds")
dk.core.save_json(CONFIG_RUN, os.path.join(P["A"], "config_run.json"))
with open(os.path.join(P["A_tab"], "environment.txt"), "w") as f:
    f.write(dk.core.env_report())
print(open(os.path.join(P["A_tab"], "environment.txt")).read())
LOG("Konfigurasi dan environment tersimpan.")

## A1. Audit dataset

### A1.1 Mengambil repo dan objek Git LFS

Citra di repo disimpan melalui Git LFS. Urutan pengambilan:

1. Bila `REPO_LOKAL` diisi, folder itu yang dipakai.
2. Bila *cache* `01_data_cache/yolo_derma_repo_cache.tar` sudah ada di Drive, *cache* diekstrak (tanpa unduh ulang).
3. Bila belum, repo di-*clone* (`--depth 1`, LFS ditunda), lalu `git lfs pull`. Berkas yang masih berupa *pointer* LFS diunduh lewat `media.githubusercontent.com` sebagai cadangan.
4. Setelah lengkap, repo (tanpa `.git/lfs`) disimpan sebagai tar di Drive beserta *commit hash*-nya, sehingga notebook 01 dan 02 memakai salinan data yang identik.

In [ ]:
CACHE_TAR = os.path.join(P["data"], "yolo_derma_repo_cache.tar")
CACHE_INFO = os.path.join(P["data"], "yolo_derma_repo_cache.json")
REPO_DIR = REPO_LOKAL or os.path.join(WORK, "YOLO-DERMA")

def daftar_pointer_lfs(folder):
    out = []
    for root, dirs, files in os.walk(folder):
        dirs[:] = [d for d in dirs if d != ".git"]
        for fn in files:
            if fn.lower().endswith(dk.data.IMG_EXT):
                p = os.path.join(root, fn)
                if dk.data.is_lfs_pointer(p):
                    out.append(p)
    return out

def unduh_media_github(paths, repo_dir):
    import requests
    from concurrent.futures import ThreadPoolExecutor
    owner_repo = REPO_URL.rstrip("/").removesuffix(".git").split("github.com/")[-1]
    def _get(p):
        rel = os.path.relpath(p, repo_dir).replace(os.sep, "/")
        url = f"https://media.githubusercontent.com/media/{owner_repo}/{REPO_BRANCH}/{rel}"
        for _ in range(3):
            try:
                r = requests.get(url, timeout=60)
                if r.status_code == 200 and not r.content.startswith(b"version https://git-lfs"):
                    with open(p, "wb") as f:
                        f.write(r.content)
                    return True
            except Exception:
                time.sleep(2)
        return False
    with ThreadPoolExecutor(16) as ex:
        ok = list(ex.map(_get, paths))
    return int(sum(ok))

repo_commit = None
if REPO_LOKAL:
    print("Memakai repo lokal:", REPO_LOKAL)
elif not os.path.exists(os.path.join(REPO_DIR, "train")) and os.path.exists(CACHE_TAR):
    print("Ekstrak cache dari Drive ...")
    with tarfile.open(CACHE_TAR) as t:
        t.extractall(WORK)
    repo_commit = dk.core.load_json(CACHE_INFO).get("commit") if os.path.exists(CACHE_INFO) else None
elif not os.path.exists(os.path.join(REPO_DIR, "train")):
    if shutil.which("git-lfs") is None:
        dk.core.run_cmd("apt-get -qq install -y git-lfs", check=False)
    dk.core.run_cmd("git lfs install --skip-repo", check=False)
    env = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")
    dk.core.run_cmd(f"git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} '{REPO_DIR}'", env=env)
    dk.core.run_cmd(f"cd '{REPO_DIR}' && git lfs pull", check=False, tail=5)

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    repo_commit = subprocess.run(f"cd '{REPO_DIR}' && git rev-parse HEAD", shell=True,
                                 capture_output=True, text=True).stdout.strip() or repo_commit

sisa = daftar_pointer_lfs(REPO_DIR)
if sisa:
    print(f"{len(sisa)} berkas masih pointer LFS, mengunduh lewat media.githubusercontent.com ...")
    print("berhasil:", unduh_media_github(sisa, REPO_DIR))
    sisa = daftar_pointer_lfs(REPO_DIR)
print("Pointer LFS tersisa:", len(sisa))

if not REPO_LOKAL and not os.path.exists(CACHE_TAR) and len(sisa) == 0:
    with tarfile.open(CACHE_TAR, "w") as t:
        t.add(REPO_DIR, arcname=os.path.basename(REPO_DIR),
              filter=lambda ti: None if "/.git/lfs" in ti.name or ti.name.endswith("/.git/lfs") else ti)
    dk.core.save_json({"commit": repo_commit, "url": REPO_URL, "branch": REPO_BRANCH,
                       "dibuat": datetime.datetime.now().isoformat(timespec="seconds")}, CACHE_INFO)
    print("Cache tersimpan:", CACHE_TAR)
print("Repo :", REPO_DIR)
print("Commit:", repo_commit)

### A1.2 *Manifest* dataset

Setiap citra mendapat pengenal permanen `sample_id = IMG_<12 karakter awal SHA-256>`. Pengenal ini tidak bergantung pada nama atau lokasi berkas, sehingga tetap sama walaupun repo direorganisasi. *Manifest* mencatat lokasi relatif, nama sumber, indeks augmentasi, waktu screenshot, resolusi, mode warna, ukuran berkas, status keterbacaan, SHA-256, pHash, dHash, keberadaan berkas label, jumlah kotak per ID kelas mentah, dan baris label yang rusak.

Berkas label dicari dengan mengganti segmen `/images/` menjadi `/labels/` (dengan atau tanpa subfolder pembagi seperti `0-500/`).

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def cari_label(img_path):
    stem = os.path.splitext(os.path.basename(img_path))[0]
    parts = img_path.split(os.sep)
    if "images" not in parts:
        return None
    i = len(parts) - 1 - parts[::-1].index("images")
    base = os.sep.join(parts[:i] + ["labels"])
    sub = parts[i + 1:-1]
    for cand in [os.path.join(base, *sub, stem + ".txt"), os.path.join(base, stem + ".txt")]:
        if os.path.exists(cand):
            return cand
    return None

semua_citra = [p for p in dk.data.list_images(REPO_DIR) if f"{os.sep}.git{os.sep}" not in p]
print("Jumlah berkas citra:", len(semua_citra))
t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    recs = list(ex.map(dk.data.image_record, semua_citra))
print(f"Hash selesai dalam {time.time() - t0:.0f} detik")

rows = []
for rec in recs:
    p = rec["path"]
    stem = os.path.splitext(os.path.basename(p))[0]
    src = dk.data.parse_source(stem)
    lp = cari_label(p)
    arr, bad = dk.data.read_yolo_label(lp) if lp else (np.zeros((0, 5)), 0)
    ids = arr[:, 0].astype(int) if len(arr) else np.zeros(0, int)
    shot = dk.data.screenshot_time(src["source_stem"])
    row = {"file_rel": os.path.relpath(p, REPO_DIR), "folder": os.path.relpath(os.path.dirname(p), REPO_DIR),
           "stem": stem, "aug_idx": src["aug_idx"], "source_stem": src["source_stem"],
           "base_source": dk.data.base_source(src["source_stem"]),
           "shot_time": shot.isoformat() if shot else None,
           "width": rec["width"], "height": rec["height"], "mode": rec["mode"], "file_bytes": rec["file_bytes"],
           "readable": rec["readable"], "err": rec["err"], "sha256": rec["sha256"],
           "phash": f"{rec['phash']:016x}" if rec["phash"] is not None else None,
           "dhash": f"{rec['dhash']:016x}" if rec["dhash"] is not None else None,
           "label_rel": os.path.relpath(lp, REPO_DIR) if lp else None, "label_exists": lp is not None,
           "n_boxes_raw": int(len(arr)), "n_bad_rows": int(bad)}
    for k in range(int(ids.max()) + 1 if len(ids) else 0):
        row[f"raw_id_{k}"] = int((ids == k).sum())
    rows.append(row)
manifest = pd.DataFrame(rows)
for c in [c for c in manifest.columns if c.startswith("raw_id_")]:
    manifest[c] = manifest[c].fillna(0).astype(int)

manifest["sample_id"] = "IMG_" + manifest["sha256"].str[:12]
dup_rank = manifest.groupby("sample_id").cumcount()
manifest.loc[dup_rank > 0, "sample_id"] = manifest.loc[dup_rank > 0, "sample_id"] + "_" + (dup_rank[dup_rank > 0] + 1).astype(str)
manifest = manifest[["sample_id"] + [c for c in manifest.columns if c != "sample_id"]]

if MODE_UJI_CEPAT:
    rng = np.random.default_rng(SEED)
    pilih = rng.choice(manifest["base_source"].unique(), size=min(N_SUMBER_UJI, manifest["base_source"].nunique()), replace=False)
    manifest = manifest[manifest["base_source"].isin(pilih)].reset_index(drop=True)
    print("MODE_UJI_CEPAT: subset", len(manifest), "citra dari", len(pilih), "sumber")

manifest.to_csv(os.path.join(P["A_tab"], "dataset_manifest.csv"), index=False)
display(manifest.head(3))
print("Tersimpan: dataset_manifest.csv")

### A1.3 Duplikat persis, *near-duplicate*, dan citra rusak

- **Duplikat persis**: SHA-256 identik.
- ***Near-duplicate***: jarak Hamming pHash **dan** dHash keduanya ≤ `HASH_HAMMING_THR` (bawaan 8 dari 64 bit). Aturan konjungtif dipakai karena pada dataset ini satu *hash* saja menghasilkan banyak pasangan palsu: pHash terpengaruh sudut hitam hasil rotasi augmentasi, dan dHash terpengaruh kulit halus bergradien rendah. Pemeriksaan visual pada uji awal menunjukkan pasangan lintas sumber yang lolos satu *hash* saja adalah citra berbeda. Jumlah pasangan menurut tiap *hash* tunggal tetap dicatat sebagai informasi. Pasangan dikategorikan *sumber sama* (varian dari screenshot yang sama, diharapkan ada) atau *lintas sumber* (dua screenshot berbeda yang nyaris identik; berisiko *leakage* bila terpisah ke *split* berbeda) dan ditampilkan dalam montase untuk diperiksa manual.
- **Citra rusak**: gagal `PIL.Image.verify()`/`load()` atau masih *pointer* LFS.

Salinan kedua dan seterusnya dari duplikat persis dikeluarkan dari evaluasi (`lolos_audit = False`) karena tidak menambah informasi. Hasil disimpan di `duplicate_report.csv`. Pasangan *lintas sumber* dipakai kembali di A3 untuk menyatukan kelompok.

In [ ]:
ph = [int(h, 16) if isinstance(h, str) else None for h in manifest["phash"]]
dh = [int(h, 16) if isinstance(h, str) else None for h in manifest["dhash"]]
def ham(a, b):
    return int(dk.data._popcount64(np.uint64(a) ^ np.uint64(b))) if a is not None and b is not None else None
tunggal = {}
for jenis, hs in [("phash", ph), ("dhash", dh)]:
    tunggal[jenis] = {(i, j) for i, j, _ in dk.data.near_duplicate_pairs(hs, thr=HASH_HAMMING_THR)}
pasangan = {ij: {"near_duplicate_phash_dhash"} for ij in tunggal["phash"] & tunggal["dhash"]}
dup_rows = []
for (i, j), jenis in pasangan.items():
    a, b = manifest.iloc[i], manifest.iloc[j]
    dup_rows.append({"jenis": "+".join(sorted(jenis)), "sample_id_a": a.sample_id, "sample_id_b": b.sample_id,
                     "file_a": a.file_rel, "file_b": b.file_rel, "hamming_phash": ham(ph[i], ph[j]), "hamming_dhash": ham(dh[i], dh[j]),
                     "base_source_a": a.base_source, "base_source_b": b.base_source,
                     "kategori": "sumber_sama" if a.base_source == b.base_source else "lintas_sumber"})
for sha, g in manifest[manifest["sha256"] != ""].groupby("sha256"):
    if len(g) > 1:
        ids = g["sample_id"].tolist()
        for k in range(1, len(ids)):
            dup_rows.append({"jenis": "exact_sha256", "sample_id_a": ids[0], "sample_id_b": ids[k],
                             "file_a": g.iloc[0].file_rel, "file_b": g.iloc[k].file_rel, "hamming_phash": 0,
                             "hamming_dhash": 0, "base_source_a": g.iloc[0].base_source,
                             "base_source_b": g.iloc[k].base_source,
                             "kategori": "sumber_sama" if g.iloc[0].base_source == g.iloc[k].base_source else "lintas_sumber"})
dup_report = pd.DataFrame(dup_rows, columns=["jenis", "sample_id_a", "sample_id_b", "file_a", "file_b", "hamming_phash",
                                             "hamming_dhash", "base_source_a", "base_source_b", "kategori"])
dup_report.to_csv(os.path.join(P["A_tab"], "duplicate_report.csv"), index=False)

salinan_dup = set(dup_report.loc[dup_report.jenis == "exact_sha256", "sample_id_b"])
manifest["exact_dup_copy"] = manifest["sample_id"].isin(salinan_dup)
manifest["lolos_audit"] = manifest["readable"] & manifest["label_exists"] & (manifest["n_bad_rows"] == 0) & ~manifest["exact_dup_copy"]
manifest.to_csv(os.path.join(P["A_tab"], "dataset_manifest.csv"), index=False)

rusak = manifest[~manifest["readable"]]
near = dup_report[dup_report.jenis != "exact_sha256"]
n_exact = int((dup_report["jenis"] == "exact_sha256").sum())
n_near = len(near)
n_near_lintas = int((near["kategori"] == "lintas_sumber").sum())
print("Citra rusak/tidak terbaca        :", len(rusak))
print("Duplikat persis (salinan)        :", len(salinan_dup))
print("Near-duplicate pHash/dHash       :", n_near, "(lintas sumber:", n_near_lintas, ")")
print("  info satu hash saja (tidak dipakai): pHash", len(tunggal["phash"] - tunggal["dhash"]), "| dHash", len(tunggal["dhash"] - tunggal["phash"]))
print("Citra lolos audit                :", int(manifest["lolos_audit"].sum()))
print("Tersimpan: duplicate_report.csv")

lintas = near[near.kategori == "lintas_sumber"].assign(skor=lambda d: d.hamming_phash + d.hamming_dhash).sort_values("skor").head(8)
if len(lintas):
    fig, ax = plt.subplots(len(lintas), 2, figsize=(7, 3.3 * len(lintas)), squeeze=False)
    for k, r in enumerate(lintas.itertuples()):
        for c, f in enumerate([r.file_a, r.file_b]):
            ax[k, c].imshow(dk.data.read_rgb(os.path.join(REPO_DIR, f))); ax[k, c].axis("off")
        ax[k, 0].set_title(f"pHash {r.hamming_phash}, dHash {r.hamming_dhash}", loc="left", fontsize=12)
    dk.viz.show_and_save(fig, os.path.join(P["A_fig"], "A1_near_duplicate_lintas_sumber"))
    print("Periksa montase: pasangan yang ternyata berbeda tetap aman (hanya membuat kelompok lebih besar).")

### A1.4 Ringkasan visual audit

Gambar A1 menampilkan empat aspek: jumlah kotak per ID kelas mentah, jumlah citra menurut ID tertinggi pada citra, sebaran jumlah kotak per citra, dan jumlah varian augmentasi per screenshot sumber. Sebaran ini menunjukkan seberapa besar ketergantungan antar-citra yang harus ditangani pada *split* dan *bootstrap*.

In [ ]:
raw_cols = sorted([c for c in manifest.columns if c.startswith("raw_id_")], key=lambda c: int(c.split("_")[-1]))
box_per_id = manifest[raw_cols].sum()
img_max_id = manifest[raw_cols].apply(lambda r: max([int(c.split("_")[-1]) for c in raw_cols if r[c] > 0], default=-1), axis=1)
var_per_src = manifest.groupby("base_source").size()

fig, ax = plt.subplots(2, 2, figsize=(13, 9))
lbl = [f"ID {c.split('_')[-1]}\n{NAMA_ID_MENTAH.get(int(c.split('_')[-1]), '?')}" for c in raw_cols]
bars = ax[0, 0].bar(lbl, box_per_id.values, color="#2F6690")
ax[0, 0].bar_label(bars, labels=[f"{v:,}" for v in box_per_id.values], fontsize=11, padding=2)
ax[0, 0].set_title("(a) Bounding boxes per raw class ID"); ax[0, 0].set_ylabel("Boxes")
vc = img_max_id.value_counts().sort_index()
bars = ax[0, 1].bar([f"ID {i}" for i in vc.index], vc.values, color="#D1495B")
ax[0, 1].bar_label(bars, fontsize=11, padding=2)
ax[0, 1].set_title("(b) Images by highest raw ID"); ax[0, 1].set_ylabel("Images")
ax[1, 0].hist(manifest["n_boxes_raw"], bins=range(0, int(manifest["n_boxes_raw"].max()) + 2), color="#4C9F70")
ax[1, 0].set_title("(c) Boxes per image"); ax[1, 0].set_xlabel("Boxes"); ax[1, 0].set_ylabel("Images")
ax[1, 1].hist(var_per_src.values, bins=range(0, int(var_per_src.max()) + 2), color="#E0A030")
ax[1, 1].set_title("(d) Augmented variants per source"); ax[1, 1].set_xlabel("Variants"); ax[1, 1].set_ylabel("Sources")
dk.viz.show_and_save(fig, os.path.join(P["A_fig"], "A1_audit_dataset"))

### A1.5 Interpretasi otomatis audit dataset

Teks berikut disusun dari angka hasil audit di atas, tanpa nilai yang diasumsikan.

In [ ]:
fmt = dk.teks.fmt
res = manifest.dropna(subset=["width"])
ukuran = res.groupby(["width", "height"]).size().sort_values(ascending=False)
teks = []
teks.append(f"Audit mencakup {len(manifest):,} berkas citra dari {manifest['base_source'].nunique():,} screenshot sumber "
            f"(setelah penyatuan unduhan ganda). Sebanyak {int(manifest['readable'].sum()):,} citra terbaca dan {len(rusak):,} citra gagal dibaca.")
teks.append(f"Resolusi yang paling sering adalah {int(ukuran.index[0][0])}x{int(ukuran.index[0][1])} piksel "
            f"({ukuran.iloc[0]:,} citra); terdapat {len(ukuran)} kombinasi resolusi berbeda.")
teks.append(f"Setiap sumber memiliki {int(var_per_src.min())}-{int(var_per_src.max())} varian (median {fmt(var_per_src.median(), 0)}). "
            "Varian dari sumber yang sama tidak independen, sehingga split dan interval kepercayaan harus berbasis kelompok sumber.")
teks.append(f"Label: {int(manifest['label_exists'].sum()):,} citra memiliki berkas label, {int((~manifest['label_exists']).sum()):,} tidak; "
            f"total {int(manifest['n_boxes_raw'].sum()):,} kotak (rerata {fmt(manifest['n_boxes_raw'].mean(), 1)} per citra) "
            f"dan {int(manifest['n_bad_rows'].sum()):,} baris label rusak.")
teks.append("Jumlah kotak per ID mentah: " + ", ".join(f"ID {c.split('_')[-1]} = {int(v):,}" for c, v in box_per_id.items()) + ".")
teks.append(f"Ditemukan {len(salinan_dup)} salinan duplikat persis (dikeluarkan dari evaluasi) dan {n_near} pasangan near-duplicate "
            f"pHash/dHash (jarak <= {HASH_HAMMING_THR}); {n_near_lintas} pasangan di antaranya lintas sumber dan akan disatukan kelompoknya di A3.")
teks.append(f"Citra lolos audit (terbaca, berlabel valid, bukan salinan duplikat): {int(manifest['lolos_audit'].sum()):,}.")
print("\n\n".join(teks))
LOG("A1 selesai: manifest dan duplicate_report tersimpan.")

## A2. Verifikasi label

### A2.1 Kecocokan `data_final.yaml` dengan isi berkas label

Kita membandingkan jumlah kelas (`nc`) dan nama kelas di YAML dengan ID kelas yang benar-benar muncul di berkas label. Bila ada ID ≥ `nc`, Ultralytics akan menandai citra tersebut sebagai *corrupt* dan membuangnya dari pelatihan. Temuan ini dicatat karena memengaruhi interpretasi bobot lama.

In [ ]:
import yaml
yaml_path = os.path.join(REPO_DIR, "data_final.yaml")
yaml_info = yaml.safe_load(open(yaml_path)) if os.path.exists(yaml_path) else {}
nc_yaml = yaml_info.get("nc")
names_yaml = yaml_info.get("names")
id_teramati = [int(c.split("_")[-1]) for c in raw_cols if box_per_id[c] > 0]
id_lebih = [i for i in id_teramati if nc_yaml is not None and i >= nc_yaml]
img_terdampak = int((manifest[[f"raw_id_{i}" for i in id_lebih]].sum(axis=1) > 0).sum()) if id_lebih else 0
cek_yaml = pd.DataFrame([
    {"pemeriksaan": "nc di YAML", "nilai": nc_yaml},
    {"pemeriksaan": "names di YAML", "nilai": names_yaml},
    {"pemeriksaan": "path train di YAML", "nilai": yaml_info.get("train")},
    {"pemeriksaan": "path val di YAML", "nilai": yaml_info.get("val")},
    {"pemeriksaan": "ID kelas teramati di label", "nilai": id_teramati},
    {"pemeriksaan": "ID >= nc (tidak sesuai YAML)", "nilai": id_lebih},
    {"pemeriksaan": "citra memuat ID >= nc", "nilai": img_terdampak},
])
display(cek_yaml)
print(f"Interpretasi: YAML mendeklarasikan {nc_yaml} kelas {names_yaml}, sedangkan berkas label memuat ID {id_teramati}. "
      + (f"ID {id_lebih} berada di luar rentang YAML dan muncul pada {img_terdampak:,} citra. "
         "Bila bobot lama dilatih dengan YAML ini apa adanya, citra tersebut terbuang dari pelatihan oleh Ultralytics." if id_lebih else
         "Seluruh ID berada dalam rentang YAML."))

### A2.2 Uji konsistensi ordinal kelas secara kolorimetrik

Pemetaan ID ke tingkat keparahan tidak tertulis di repo. Sebagai bukti tambahan yang tetap memerlukan konfirmasi ahli, kita mengukur tiga ciri sederhana pada crop lesi per ID mentah:

- **a\*** (CIELAB): indeks kemerahan, lazim dipakai sebagai *proxy* eritema dalam kolorimetri kulit (Takiwaki, 1998; Clarys et al., 2000);
- **L\***: kecerahan;
- **variansi Laplacian**: kekasaran tekstur (berkaitan dengan *scaling*, papul, dan likenifikasi).

Piksel hitam hasil rotasi augmentasi (L\* < 2%) dikeluarkan dari perhitungan. Bila ID yang dipetakan ke Ringan → Sedang → Berat memperlihatkan kenaikan a\* dan tekstur yang monoton, pemetaan bawaan mendapat dukungan data. Korelasi Spearman antara urutan keparahan dan ciri tersebut dihitung dengan 95% CI *cluster bootstrap* per sumber.

In [ ]:
import cv2
from scipy.stats import spearmanr
rng = np.random.default_rng(SEED)
MAKS_KOTAK_PER_ID = 400 if MODE_UJI_CEPAT else 2500
feat_rows = []
by_id = {}
for r in manifest[manifest["label_exists"] & manifest["readable"]].itertuples():
    arr, _ = dk.data.read_yolo_label(os.path.join(REPO_DIR, r.label_rel))
    for b, a in enumerate(arr):
        by_id.setdefault(int(a[0]), []).append((r.Index, b))
for cid, lst in by_id.items():
    sel = [lst[i] for i in rng.permutation(len(lst))[:MAKS_KOTAK_PER_ID]]
    cache = {}
    for idx, b in sel:
        r = manifest.loc[idx]
        if idx not in cache:
            cache.clear()
            cache[idx] = (dk.data.read_rgb(os.path.join(REPO_DIR, r.file_rel)),
                          dk.data.read_yolo_label(os.path.join(REPO_DIR, r.label_rel))[0])
        img, arr = cache[idx]
        if img is None:
            continue
        H, W = img.shape[:2]
        x1, y1, x2, y2 = dk.data.yolo_to_xyxy(arr[b:b + 1, 1:5], W, H)[0].astype(int)
        if x2 - x1 < 4 or y2 - y1 < 4:
            continue
        crop = img[y1:y2, x1:x2]
        lab = cv2.cvtColor(crop, cv2.COLOR_RGB2LAB).astype(np.float32)
        valid = lab[..., 0] > 5
        if valid.mean() < 0.5:
            continue
        gray = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
        feat_rows.append({"raw_id": cid, "base_source": r.base_source,
                          "a_star": float(lab[..., 1][valid].mean() - 128), "L_star": float(lab[..., 0][valid].mean() * 100 / 255),
                          "laplacian_var": float(cv2.Laplacian(gray, cv2.CV_64F).var()),
                          "area_frac": float((x2 - x1) * (y2 - y1) / (W * H))})
feat = pd.DataFrame(feat_rows)
ringkas = feat.groupby("raw_id").agg(n=("a_star", "size"),
                                     a_star_median=("a_star", "median"), a_star_q1=("a_star", lambda s: s.quantile(.25)),
                                     a_star_q3=("a_star", lambda s: s.quantile(.75)), L_star_median=("L_star", "median"),
                                     laplacian_median=("laplacian_var", "median"), area_frac_median=("area_frac", "median")).reset_index()
ringkas["nama_hipotesis"] = ringkas["raw_id"].map(NAMA_ID_MENTAH)
ringkas["indeks_keparahan"] = ringkas["raw_id"].map(lambda i: LABEL_MAP.get(int(i), -1))

fm = feat[feat["raw_id"].isin(LABEL_MAP.keys())].copy()
fm["sev"] = fm["raw_id"].map(LABEL_MAP)
rho = {}
cb = dk.stats.ClusterBootstrap(B=200, seed=SEED).register("f", fm["base_source"].values)
for col in ["a_star", "laplacian_var", "L_star"]:
    est = spearmanr(fm["sev"], fm[col]).statistic
    boots = []
    for d in cb.draws():
        s = fm.iloc[d["f"]]
        boots.append(spearmanr(s["sev"], s[col]).statistic if s["sev"].nunique() > 1 else np.nan)
    rho[col] = (est,) + dk.stats.percentile_ci(boots)
spearman_df = pd.DataFrame([{"ciri": k, "rho_spearman": v[0], "ci_lo": v[1], "ci_hi": v[2]} for k, v in rho.items()])
ringkas.to_csv(os.path.join(P["A_tab"], "label_class_stats.csv"), index=False)
spearman_df.to_csv(os.path.join(P["A_tab"], "label_ordinal_spearman.csv"), index=False)
display(ringkas.round(3)); display(spearman_df.round(3))

fig, ax = plt.subplots(1, 3, figsize=(15, 4.8))
for k, (col, judul) in enumerate([("a_star", "a* (redness)"), ("laplacian_var", "Laplacian variance (texture)"), ("L_star", "L* (lightness)")]):
    ids_f = sorted(feat.raw_id.unique())
    data = [feat.loc[feat.raw_id == i, col].values for i in ids_f]
    ax[k].boxplot(data, showfliers=False)
    ax[k].set_xticks(range(1, len(ids_f) + 1)); ax[k].set_xticklabels([f"ID {i}" for i in ids_f])
    ax[k].set_title(judul)
    if col == "laplacian_var":
        ax[k].set_yscale("log")
dk.viz.show_and_save(fig, os.path.join(P["A_fig"], "A2_ciri_kolorimetrik_per_id"))

In [ ]:
fmt, fci = dk.teks.fmt, dk.teks.fmt_ci
urut = [k for k, v in sorted(LABEL_MAP.items(), key=lambda kv: kv[1])]
med = ringkas.set_index("raw_id")
def monoton(col):
    vals = [med.loc[i, col] for i in urut if i in med.index]
    return all(np.diff(vals) > 0) if len(vals) > 1 else False
teks = []
teks.append("Median a* per ID: " + ", ".join(f"ID {int(i)} = {fmt(v, 2)}" for i, v in med["a_star_median"].items()) + ".")
teks.append("Median variansi Laplacian per ID: " + ", ".join(f"ID {int(i)} = {fmt(v, 2)}" for i, v in med["laplacian_median"].items()) + ".")
for col, nama in [("a_star_median", "a* (kemerahan)"), ("laplacian_median", "tekstur")]:
    teks.append(f"Pada urutan pemetaan {urut} (Ringan -> Sedang -> Berat), median {nama} " +
                ("naik secara monoton." if monoton(col) else "tidak naik secara monoton."))
for _, r in spearman_df.iterrows():
    teks.append(f"Spearman rho antara indeks keparahan dan {r.ciri} = {fmt(r.rho_spearman)} (95% CI {fci(r.ci_lo, r.ci_hi)}).")
teks.append("Hasil ini hanya menguji konsistensi ordinal ciri visual sederhana. Pemetaan tetap berstatus "
            f"{STATUS_LABEL_MAP} sampai anotator atau dokter mengonfirmasi arti setiap ID.")
print("\n".join(teks))

### A2.3 Montase contoh lesi per ID mentah

Montase ini disiapkan untuk diperiksa bersama anotator/dokter saat mengonfirmasi arti setiap ID. Crop diambil acak dari sumber berbeda dengan padding yang sama seperti input CNN.

In [ ]:
N_CONTOH = 10
ids_ada = sorted(by_id.keys())
fig, ax = plt.subplots(len(ids_ada), N_CONTOH, figsize=(N_CONTOH * 1.6, len(ids_ada) * 1.9))
ax = np.atleast_2d(ax)
for r_i, cid in enumerate(ids_ada):
    lst = by_id[cid]
    perm = rng.permutation(len(lst))
    used_src, shown = set(), 0
    for pi in perm:
        if shown >= N_CONTOH:
            break
        idx, b = lst[pi]
        row = manifest.loc[idx]
        if row.base_source in used_src and len(used_src) < 50:
            continue
        img = dk.data.read_rgb(os.path.join(REPO_DIR, row.file_rel))
        arr = dk.data.read_yolo_label(os.path.join(REPO_DIR, row.label_rel))[0]
        if img is None:
            continue
        H, W = img.shape[:2]
        xyxy = dk.data.yolo_to_xyxy(arr[b:b + 1, 1:5], W, H)
        crop = dk.models.crop_rois(img, xyxy, ROI_PAD, 96)[0].astype(np.uint8)
        ax[r_i, shown].imshow(crop); used_src.add(row.base_source); shown += 1
    for c in range(N_CONTOH):
        ax[r_i, c].axis("off")
    ax[r_i, 0].set_title(f"ID {cid}: {NAMA_ID_MENTAH.get(cid, '?')}", loc="left", fontsize=12)
dk.viz.show_and_save(fig, os.path.join(P["A_fig"], "A2_montase_per_id"))

### A2.4 Penerapan pemetaan label dan pemeriksaan kotak

Setiap kotak diperiksa dan ditandai:

- `flag_oob`: koordinat ternormalisasi di luar [0, 1] sebelum dipotong;
- `flag_degenerate`: lebar atau tinggi ≤ 0;
- `flag_tiny`: sisi terpendek < `MIN_SISI_KOTAK_PX` piksel;
- `flag_dup`: kotak dengan ID sama yang tumpang tindih IoU > `IOU_KOTAK_GANDA` dengan kotak sebelumnya (anotasi ganda).

Kotak dipakai sebagai lesi referensi bila `keep = True` (ID dipetakan dan tidak *degenerate/tiny/dup*). Kasus multi-lesi dicatat per citra: jumlah lesi, kelas maksimum (aturan tingkat citra), kelas mayoritas, dan penanda campuran kelas. Hasil: `verified_labels.csv` dan `label_mapping.json`.

In [ ]:
vrows = []
for r in manifest[manifest["label_exists"]].itertuples():
    arr, _ = dk.data.read_yolo_label(os.path.join(REPO_DIR, r.label_rel))
    if len(arr) == 0:
        continue
    W, H = float(r.width), float(r.height)
    xyxy = dk.data.yolo_to_xyxy(arr[:, 1:5], W, H)
    oob = ((arr[:, 1:5] < 0) | (arr[:, 1:5] > 1)).any(axis=1)
    iou = dk.geometry.iou_matrix(xyxy, xyxy)
    for b in range(len(arr)):
        raw = int(arr[b, 0])
        w_px, h_px = xyxy[b, 2] - xyxy[b, 0], xyxy[b, 3] - xyxy[b, 1]
        same = (arr[:b, 0].astype(int) == raw)
        dup = bool((iou[b, :b][same] > IOU_KOTAK_GANDA).any()) if b > 0 else False
        sev = LABEL_MAP.get(raw, -1)
        degenerate = bool(w_px <= 0 or h_px <= 0)
        tiny = bool(min(w_px, h_px) < MIN_SISI_KOTAK_PX)
        vrows.append({"sample_id": r.sample_id, "box_idx": b, "raw_id": raw, "sev": sev,
                      "xc": arr[b, 1], "yc": arr[b, 2], "w": arr[b, 3], "h": arr[b, 4],
                      "x1": xyxy[b, 0], "y1": xyxy[b, 1], "x2": xyxy[b, 2], "y2": xyxy[b, 3],
                      "w_px": w_px, "h_px": h_px, "area_frac": w_px * h_px / (W * H),
                      "flag_oob": bool(oob[b]), "flag_degenerate": degenerate, "flag_tiny": tiny, "flag_dup": dup,
                      "keep": bool(sev >= 0 and not degenerate and not tiny and not dup)})
ver = pd.DataFrame(vrows)
ver.to_csv(os.path.join(P["A_tab"], "verified_labels.csv"), index=False)

kept = ver[ver["keep"]]
img_lvl = kept.groupby("sample_id")["sev"].agg(n_lesi="size", sev_max="max",
                                               sev_majority=lambda s: int(s.value_counts().idxmax()),
                                               n_kelas=lambda s: s.nunique()).reset_index()
manifest = manifest.drop(columns=[c for c in ["n_lesi", "sev_max", "sev_majority", "n_kelas"] if c in manifest.columns])
manifest = manifest.merge(img_lvl, on="sample_id", how="left")
manifest["n_lesi"] = manifest["n_lesi"].fillna(0).astype(int)
manifest["sev_max"] = manifest["sev_max"].fillna(-1).astype(int)
manifest["sev_majority"] = manifest["sev_majority"].fillna(-1).astype(int)
manifest["n_kelas"] = manifest["n_kelas"].fillna(0).astype(int)
manifest.to_csv(os.path.join(P["A_tab"], "dataset_manifest.csv"), index=False)

label_mapping = {
    "label_map_raw_to_severity": {str(k): v for k, v in LABEL_MAP.items()},
    "severity_names_id": NAMA_KELAS_ID, "severity_names_en": dk.core.class_names("en"),
    "raw_id_names_hypothesis": {str(k): v for k, v in NAMA_ID_MENTAH.items()},
    "status": STATUS_LABEL_MAP,
    "yaml": {"nc": nc_yaml, "names": names_yaml, "train": yaml_info.get("train"), "val": yaml_info.get("val")},
    "raw_ids_observed": id_teramati,
    "ordinal_evidence_spearman": spearman_df.to_dict(orient="records"),
    "rules": {"image_level_label": "keparahan maksimum di antara lesi (keep=True)",
              "excluded_box_flags": ["flag_degenerate", "flag_tiny", "flag_dup", "raw_id tidak dipetakan"],
              "min_side_px": MIN_SISI_KOTAK_PX, "dup_iou": IOU_KOTAK_GANDA},
    "yolo_target": "satu kelas 'lesi' dari kotak keep=True (YOLO hanya melokalisasi, CNN menilai keparahan)",
}
dk.core.save_json(label_mapping, os.path.join(P["A_tab"], "label_mapping.json"))

ringkas_flag = pd.DataFrame({
    "jumlah_kotak": [len(ver)], "dipetakan": [int((ver.sev >= 0).sum())],
    "tidak_dipetakan": [int((ver.sev < 0).sum())], "oob": [int(ver.flag_oob.sum())],
    "degenerate": [int(ver.flag_degenerate.sum())], "tiny": [int(ver.flag_tiny.sum())],
    "dup": [int(ver.flag_dup.sum())], "keep": [int(ver.keep.sum())]})
display(ringkas_flag)
print("Tersimpan: verified_labels.csv, label_mapping.json")

In [ ]:
fmt, pct = dk.teks.fmt, dk.teks.pct
lesi = manifest[manifest.n_lesi > 0]
teks = []
teks.append(f"Dari {len(ver):,} kotak, {int((ver.sev >= 0).sum()):,} memiliki ID yang dipetakan ke tingkat keparahan dan "
            f"{int(ver.keep.sum()):,} lolos pemeriksaan (keep). Kotak yang dikeluarkan: {int(ver.flag_tiny.sum()):,} terlalu kecil, "
            f"{int(ver.flag_dup.sum()):,} anotasi ganda, {int(ver.flag_degenerate.sum()):,} degenerate; {int(ver.flag_oob.sum()):,} kotak keluar batas citra sebelum dipotong.")
teks.append("Lesi per kelas keparahan (keep): " + ", ".join(f"{NAMA_KELAS_ID[k]} = {int((kept.sev == k).sum()):,}" for k in range(3)) + ".")
teks.append(f"{len(lesi):,} citra memiliki minimal satu lesi terverifikasi; {int((manifest.n_lesi == 0).sum()):,} citra tanpa lesi terverifikasi "
            "(dipakai sebagai citra latar/negatif untuk YOLO).")
teks.append(f"Median lesi per citra = {fmt(lesi.n_lesi.median(), 0)} (maksimum {int(lesi.n_lesi.max()) if len(lesi) else 0}). "
            f"{pct((lesi.n_kelas > 1).mean())} citra berlesi memuat lebih dari satu kelas keparahan, sehingga unit analisis utama "
            "adalah lesi (ROI), dan label tingkat citra memakai aturan keparahan maksimum.")
teks.append("Label tingkat citra (maksimum): " + ", ".join(f"{NAMA_KELAS_ID[k]} = {int((manifest.sev_max == k).sum()):,}" for k in range(3)) + ".")
print("\n".join(teks))
LOG("A2 selesai: verified_labels dan label_mapping tersimpan.")

## A3. *Split* per kelompok dan audit *data leakage*

### A3.1 Pembentukan kelompok anti-*leakage*

`patient_id` tidak tersedia. Sebagai pengganti, kelompok dibentuk dengan *union-find* atas empat relasi:

1. citra dari **screenshot sumber yang sama** (semua varian augmentasi);
2. **unduhan ganda** dengan nama dasar sama (`X` dan `X-2-`);
3. **sesi pengambilan**: screenshot dengan selang waktu ≤ `SESSION_GAP_SEC` detik dari screenshot sebelumnya (screenshot beruntun berpotensi berasal dari video atau bayi yang sama);
4. ***near-duplicate* lintas sumber** (pHash atau dHash) dari A1.3.

Hanya citra `lolos_audit` yang masuk pengelompokan dan *split*. Kelompok ini menjadi unit *split* dan unit *resampling* pada *bootstrap*.

In [ ]:
man_ok = manifest[manifest["lolos_audit"]].copy()
uf = dk.data.UnionFind(man_ok["sample_id"].tolist())
rep = man_ok.groupby("base_source")["sample_id"].first()
for bs, g in man_ok.groupby("base_source"):
    ids = g["sample_id"].tolist()
    for s in ids[1:]:
        uf.union(ids[0], s)
times = man_ok.dropna(subset=["shot_time"]).groupby("base_source")["shot_time"].first().sort_values()
tt = pd.to_datetime(times.values)
n_sesi_link = 0
for i in range(1, len(tt)):
    if (tt[i] - tt[i - 1]).total_seconds() <= SESSION_GAP_SEC:
        uf.union(rep[times.index[i - 1]], rep[times.index[i]]); n_sesi_link += 1
n_dup_link = 0
for r in dup_report[dup_report.kategori == "lintas_sumber"].itertuples():
    if r.sample_id_a in uf.parent and r.sample_id_b in uf.parent:
        uf.union(r.sample_id_a, r.sample_id_b); n_dup_link += 1
root = uf.groups()
roots = sorted(set(root.values()))
gid = {r: f"G{i:04d}" for i, r in enumerate(roots)}
man_ok["group_id"] = man_ok["sample_id"].map(lambda s: gid[root[s]])
grp = man_ok.groupby("group_id").agg(n_img=("sample_id", "size"), n_src=("base_source", "nunique"), sev_max=("sev_max", "max"))
print(f"Citra lolos audit: {len(man_ok):,}, sumber unik: {man_ok.base_source.nunique()}, kelompok akhir: {len(grp)}")
print(f"Tautan sesi: {n_sesi_link}, tautan near-duplicate lintas sumber: {n_dup_link}")
print("Ukuran kelompok (jumlah sumber):", grp.n_src.describe()[["min", "50%", "max"]].to_dict())

### A3.2 *Split* bertingkat per kelompok dan penguncian

Pembagian memakai `StratifiedGroupKFold` (20 lipatan, *seed* tetap) dengan strata = kelas keparahan maksimum kelompok (kelompok tanpa lesi menjadi strata tersendiri). Lipatan dialokasikan ke *train/val/test* sesuai `SPLIT_PROPORSI`. Hasil dikunci di `locked_data_split.csv` beserta `split_lock.json` (SHA-256 isi *split*). Pada eksekusi berikutnya berkas terkunci dibaca ulang dan dicocokkan dengan SHA-256 citra; *split* hanya dibuat ulang bila `FORCE_RESPLIT = True`.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
SPLIT_CSV = os.path.join(P["A_tab"], "locked_data_split.csv")
LOCK_JSON = os.path.join(P["A_tab"], "split_lock.json")
cols_split = ["sample_id", "file_rel", "sha256", "group_id", "base_source", "split", "sev_max", "n_lesi", "width", "height"]

def hash_split(df):
    return dk.core.sha256_text("\n".join(f"{a},{b}" for a, b in sorted(zip(df.sample_id, df.split))))

if os.path.exists(SPLIT_CSV) and not FORCE_RESPLIT:
    split_df = pd.read_csv(SPLIT_CSV)
    cocok = set(split_df.sha256) == set(man_ok.sha256)
    print("Split terkunci dibaca ulang. SHA-256 cocok dengan manifest:", cocok)
    if not cocok:
        raise RuntimeError("Isi data berubah dibanding split terkunci. Periksa repo atau set FORCE_RESPLIT=True secara sadar.")
    man_ok = man_ok.drop(columns=["group_id"]).merge(split_df[["sample_id", "group_id", "split"]], on="sample_id", how="left")
else:
    strata = grp["sev_max"].clip(lower=-1).astype(int) + 1
    n_folds = 20 if len(grp) >= 40 else max(4, len(grp) // 4)
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    gdf = grp.reset_index()
    fold_of = {}
    for f, (_, te) in enumerate(sgkf.split(gdf, strata.values, groups=gdf["group_id"].values)):
        for i in te:
            fold_of[gdf.loc[i, "group_id"]] = f
    n_te = max(1, int(round(SPLIT_PROPORSI[2] * n_folds))); n_va = max(1, int(round(SPLIT_PROPORSI[1] * n_folds)))
    n_tr = n_folds - n_va - n_te
    fold_split = {f: ("train" if f < n_tr else "val" if f < n_tr + n_va else "test") for f in range(n_folds)}
    man_ok["split"] = man_ok["group_id"].map(lambda g: fold_split[fold_of[g]])
    split_df = man_ok[cols_split].copy()
    split_df.to_csv(SPLIT_CSV, index=False)
    dk.core.save_json({"sha256_split": hash_split(split_df), "seed": SEED, "n_folds": n_folds,
                       "proporsi": SPLIT_PROPORSI, "session_gap_sec": SESSION_GAP_SEC, "phash_thr": HASH_HAMMING_THR,
                       "repo_commit": repo_commit, "dibuat": datetime.datetime.now().isoformat(timespec="seconds")}, LOCK_JSON)
    print("Split baru dibuat dan dikunci.")
split_df = pd.read_csv(SPLIT_CSV)
print("SHA-256 split:", hash_split(split_df))

### A3.3 Audit *data leakage*

Pemeriksaan yang dicatat di `leakage_audit.csv`:

| Pemeriksaan | Harapan |
|---|---|
| kelompok yang muncul di lebih dari satu *split* | 0 |
| sumber screenshot yang muncul di lebih dari satu *split* | 0 |
| SHA-256 identik lintas *split* | 0 |
| *near-duplicate* pHash atau dHash lintas *split* | 0 |
| sumber yang berdekatan waktu (≤ 2 × `SESSION_GAP_SEC`) tetapi berbeda *split* | informatif; sesi bertetangga diperiksa bila jumlahnya besar |
| status bobot lama terhadap data repo | *in-sample* bila YAML menunjuk folder ini sebagai data latih |

In [ ]:
m = manifest.merge(split_df[["sample_id"]], on="sample_id")
sp = dict(zip(split_df.sample_id, split_df.split))
def lintas(col):
    return int((split_df.groupby(col)["split"].nunique() > 1).sum())
cek = []
cek.append(("kelompok lintas split", lintas("group_id"), 0))
cek.append(("sumber screenshot lintas split", lintas("base_source"), 0))
cek.append(("SHA-256 identik lintas split", lintas("sha256"), 0))
near_l = dup_report[dup_report.jenis != "exact_sha256"]
cek.append(("near-duplicate pHash/dHash lintas split",
            int(sum(sp.get(a) != sp.get(b) for a, b in zip(near_l.sample_id_a, near_l.sample_id_b) if a in sp and b in sp)), 0))
src_split = split_df.groupby("base_source")["split"].first()
tsrc = man_ok.dropna(subset=["shot_time"]).groupby("base_source")["shot_time"].first().sort_values()
tts = pd.to_datetime(tsrc.values)
n_tetangga = int(sum((tts[i] - tts[i - 1]).total_seconds() <= 2 * SESSION_GAP_SEC and src_split[tsrc.index[i]] != src_split[tsrc.index[i - 1]]
                     for i in range(1, len(tts))))
yaml_train = str(yaml_info.get("train", ""))
status_lama = "IN-SAMPLE: YAML menunjuk dataset_10000/train sebagai data latih bobot lama" if "10000" in yaml_train or "train" in yaml_train else "TIDAK DIKETAHUI"
leak = pd.DataFrame([{"pemeriksaan": a, "nilai": b, "harapan": c, "status": "OK" if b == c else "PERINGATAN"} for a, b, c in cek])
leak = pd.concat([leak, pd.DataFrame([
    {"pemeriksaan": f"sumber bertetangga waktu (<= {2 * SESSION_GAP_SEC} detik) beda split", "nilai": n_tetangga,
     "harapan": "informatif", "status": "INFO"},
    {"pemeriksaan": "status bobot lama terhadap seluruh citra repo", "nilai": yaml_train,
     "harapan": "tidak pernah dilihat model", "status": status_lama}])], ignore_index=True)
leak.to_csv(os.path.join(P["A_tab"], "leakage_audit.csv"), index=False)
display(leak)

### A3.4 Ringkasan *split* dan bagian internal Tabel 4

Tabel ringkasan memuat jumlah citra, sumber, kelompok, lesi per kelas, dan label tingkat citra per *split*. Bagian internal Tabel 4 (karakteristik dataset) disimpan sebagai `tabel4_internal.csv` dan digabung dengan data eksternal di notebook 01.

In [ ]:
kept_s = kept.merge(split_df[["sample_id", "split"]], on="sample_id")
rows = []
for s in ["train", "val", "test"]:
    d = split_df[split_df.split == s]
    k = kept_s[kept_s.split == s]
    row = {"split": s, "citra": len(d), "sumber": d.base_source.nunique(), "kelompok": d.group_id.nunique(),
           "citra_tanpa_lesi": int((d.n_lesi == 0).sum())}
    for c in range(3):
        row[f"lesi_{NAMA_KELAS_ID[c]}"] = int((k.sev == c).sum())
        row[f"citra_{NAMA_KELAS_ID[c]}(maks)"] = int((d.sev_max == c).sum())
    rows.append(row)
split_summary = pd.DataFrame(rows)
split_summary.to_csv(os.path.join(P["A_tab"], "split_summary.csv"), index=False)
display(split_summary)

lolos = manifest[manifest.lolos_audit]
res_min = f"{int(lolos.width.min())}x{int(lolos.height.min())}" if len(lolos) else "NA"
res_max = f"{int(lolos.width.max())}x{int(lolos.height.max())}" if len(lolos) else "NA"
n_dup_img = len(salinan_dup)
n_label_invalid = int(((manifest.n_bad_rows > 0) | (~manifest.label_exists)).sum())
tab4 = pd.DataFrame([
    ("Total citra sebelum audit", len(manifest)), ("Citra duplikat persis", n_dup_img),
    ("Citra dengan label tidak valid/tidak ada", n_label_invalid), ("Citra lolos audit", len(lolos)),
    ("Sumber screenshot unik (lolos audit)", lolos.base_source.nunique()), ("Kelompok anti-leakage", split_df.group_id.nunique()),
    ("Pasien unik", "NA"),
    ("Citra Ringan (maks)", int((lolos.sev_max == 0).sum())), ("Citra Sedang (maks)", int((lolos.sev_max == 1).sum())),
    ("Citra Berat (maks)", int((lolos.sev_max == 2).sum())), ("Citra tanpa lesi terverifikasi", int((lolos.sev_max < 0).sum())),
    ("Lesi Ringan", int((kept.sev == 0).sum())), ("Lesi Sedang", int((kept.sev == 1).sum())), ("Lesi Berat", int((kept.sev == 2).sum())),
    ("Rentang resolusi", f"{res_min} - {res_max}"), ("Citra test (split terkunci)", int((split_df.split == "test").sum())),
], columns=["Karakteristik", "Dataset Internal"])
tab4.to_csv(os.path.join(P["A_tab"], "tabel4_internal.csv"), index=False)
display(tab4)

In [ ]:
pct = dk.teks.pct
teks = []
for r in split_summary.itertuples():
    teks.append(f"Split {r.split}: {r.citra:,} citra dari {r.sumber} sumber dan {r.kelompok} kelompok; "
                + ", ".join(f"{NAMA_KELAS_ID[c]} {getattr(r, f'lesi_{NAMA_KELAS_ID[c]}'):,} lesi" for c in range(3)) + ".")
tot = split_summary.citra.sum()
teks.append("Proporsi citra train/val/test = " + "/".join(pct(v / tot) for v in split_summary.citra) +
            " (proporsi dihitung per kelompok, sehingga proporsi citra dapat sedikit menyimpang dari target).")
peringatan = leak[leak.status == "PERINGATAN"]
teks.append("Audit leakage: " + ("semua pemeriksaan bernilai sesuai harapan." if len(peringatan) == 0 else
            "ada pemeriksaan bertanda PERINGATAN: " + "; ".join(f"{a} = {b}" for a, b in zip(peringatan.pemeriksaan, peringatan.nilai))))
teks.append(f"Status bobot lama: {status_lama}. Hasil bobot lama pada split mana pun dari repo ini tidak dapat diklaim sebagai unseen test.")
print("\n".join(teks))
LOG("A3 selesai: split terkunci dan leakage audit tersimpan.")

## A4. Model

### A4.1 Dataset YOLO dan indeks ROI

Dataset YOLO dibangun di folder kerja lokal (`/content/work/yolo_ds`) dari *split* terkunci: citra berupa *symlink* ke repo, label ditulis ulang menjadi satu kelas `lesi` dari kotak `keep = True`. Citra tanpa lesi terverifikasi tetap disertakan dengan berkas label kosong sebagai contoh negatif. Indeks ROI per *split* (`rois_<split>.csv`) dipakai untuk pelatihan CNN.

In [ ]:
YDS = os.path.join(WORK, "yolo_ds")
if os.path.exists(YDS):
    shutil.rmtree(YDS)
kept_by = {k: g for k, g in kept.groupby("sample_id")}
for s in ["train", "val", "test"]:
    os.makedirs(f"{YDS}/{s}/images", exist_ok=True); os.makedirs(f"{YDS}/{s}/labels", exist_ok=True)
for r in split_df.itertuples():
    src = os.path.join(REPO_DIR, r.file_rel)
    ext = os.path.splitext(r.file_rel)[1]
    dst = f"{YDS}/{r.split}/images/{r.sample_id}{ext}"
    os.symlink(src, dst)
    g = kept_by.get(r.sample_id)
    with open(f"{YDS}/{r.split}/labels/{r.sample_id}.txt", "w") as f:
        if g is not None:
            for b in g.itertuples():
                f.write(f"0 {b.xc:.6f} {b.yc:.6f} {b.w:.6f} {b.h:.6f}\n")
YOLO_YAML = os.path.join(WORK, "data_yolo_lesi.yaml")
with open(YOLO_YAML, "w") as f:
    yaml.safe_dump({"path": YDS, "train": "train/images", "val": "val/images", "test": "test/images",
                    "nc": 1, "names": ["lesi"]}, f)

rois = kept.merge(split_df[["sample_id", "split", "group_id", "file_rel", "width", "height"]], on="sample_id")
rois["path"] = rois["file_rel"].map(lambda p: os.path.join(REPO_DIR, p))
for s in ["train", "val", "test"]:
    rois[rois.split == s].drop(columns=["path"]).to_csv(os.path.join(P["A_tab"], f"rois_{s}.csv"), index=False)
print(open(YOLO_YAML).read())
print(rois.groupby(["split", "sev"]).size().unstack(fill_value=0))

### A4.2 Pelatihan ulang YOLO (detektor lesi satu kelas)

Sesuai paper rujukan, YOLO hanya bertugas melokalisasi lesi; penilaian keparahan dikerjakan CNN. Parameter: arsitektur `YOLO_ARCH` dengan bobot pralatih COCO, `imgsz = 640`, *early stopping* `patience = 20`, *seed* tetap, augmentasi bawaan Ultralytics. Folder *run* berada di Drive (`A_PERSIAPAN/models/yolo_runs/retrain`). Bila sesi terputus, sel ini melanjutkan dari `last.pt`. Bila `best.pt` dan penanda selesai sudah ada, pelatihan dilewati.

Augmentasi bawaan Ultralytics mencakup HSV (*hue/saturation/value*), translasi, skala, *flip*, dan *mosaic*. Augmentasi fotometrik ini ikut memengaruhi ketahanan terhadap *brightness/contrast* di notebook 01; konfigurasinya dicatat di `model_config.json`.

In [ ]:
from ultralytics import YOLO
YRUN = os.path.join(P["A_model"], "yolo_runs")
YRUN_DIR = os.path.join(YRUN, "retrain")
Y_BEST, Y_LAST = os.path.join(YRUN_DIR, "weights", "best.pt"), os.path.join(YRUN_DIR, "weights", "last.pt")
Y_DONE = os.path.join(YRUN_DIR, "SELESAI.txt")
Y_FINAL = os.path.join(P["A_model"], "yolo_retrain_best.pt")
yolo_args = dict(data=YOLO_YAML, imgsz=YOLO_IMGSZ, epochs=YOLO_EPOCHS, patience=YOLO_PATIENCE, batch=YOLO_BATCH,
                 seed=SEED, deterministic=True, single_cls=True, project=YRUN, name="retrain", exist_ok=True,
                 workers=2, plots=True, verbose=False)
t0 = time.time()
if os.path.exists(Y_DONE) and os.path.exists(Y_BEST):
    print("Pelatihan YOLO sudah selesai sebelumnya; dilewati.")
elif os.path.exists(Y_LAST):
    print("Melanjutkan pelatihan YOLO dari last.pt ...")
    try:
        YOLO(Y_LAST).train(resume=True)
    except Exception as e:
        print("Resume gagal:", e, "-> mulai ulang dari bobot pralatih.")
        YOLO(YOLO_ARCH).train(**yolo_args)
else:
    YOLO(YOLO_ARCH).train(**yolo_args)
if os.path.exists(Y_BEST):
    shutil.copy2(Y_BEST, Y_FINAL)
    open(Y_DONE, "w").write(datetime.datetime.now().isoformat())
print(f"Durasi sel: {(time.time() - t0) / 60:.1f} menit")
res_csv = os.path.join(YRUN_DIR, "results.csv")
yolo_hist = pd.read_csv(res_csv) if os.path.exists(res_csv) else pd.DataFrame()
yolo_hist.columns = [c.strip() for c in yolo_hist.columns]
display(yolo_hist.tail(3))

In [ ]:
if len(yolo_hist):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    for c, lab in [("train/box_loss", "train box loss"), ("val/box_loss", "val box loss")]:
        if c in yolo_hist:
            ax[0].plot(yolo_hist["epoch"], yolo_hist[c], label=lab)
    ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss"); ax[0].legend()
    for c, lab in [("metrics/mAP50(B)", "mAP@0.5"), ("metrics/mAP50-95(B)", "mAP@0.5:0.95")]:
        if c in yolo_hist:
            ax[1].plot(yolo_hist["epoch"], yolo_hist[c], label=lab)
    ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Validation mAP"); ax[1].legend()
    dk.viz.show_and_save(fig, os.path.join(P["A_fig"], "A4_yolo_kurva_latih"))
    best_ep = int(yolo_hist["metrics/mAP50-95(B)"].idxmax())
    r = yolo_hist.loc[best_ep]
    print(f"Interpretasi: pelatihan berjalan {len(yolo_hist)} epoch; mAP@0.5:0.95 validasi tertinggi "
          f"{dk.teks.fmt(r['metrics/mAP50-95(B)'])} (mAP@0.5 = {dk.teks.fmt(r['metrics/mAP50(B)'])}) pada epoch {int(r['epoch'])}. "
          + ("Pelatihan berhenti lebih awal oleh early stopping." if len(yolo_hist) < YOLO_EPOCHS else "Seluruh epoch terpakai."))

### A4.3 Pelatihan ulang CNN EfficientNet-B0 (klasifikasi keparahan ROI)

Rancangan mengikuti paper rujukan: EfficientNet-B0 pralatih ImageNet, masukan 224×224, ROI dengan padding 12,5%, *weighted cross-entropy* (bobot kelas seimbang), *dropout*, *cosine decay*, dan augmentasi geometrik serta fotometrik ringan.

- **Data**: ROI referensi dari *split train* (maksimal `CNN_MAX_ROI_PER_IMG` ROI per citra, diacak dengan *seed*) untuk mengurangi redundansi antarvarian; validasi dari *split val*.
- **Crop**: `tf.image.crop_and_resize` dengan koordinat dari `boxes_to_tf_norm`, fungsi yang sama dengan inferensi.
- **Augmentasi**: *flip* horizontal/vertikal, rotasi ±10°, *zoom* ±10%, *brightness* dan *contrast* ±10%, *Gaussian noise* σ = 3 (skala 0–255).
- **Tahap 1**: *backbone* dibekukan, hanya *head* dilatih (`CNN_EPOCHS_HEAD` epoch, Adam 1e-3).
- **Tahap 2**: `CNN_FT_LAYERS` lapisan teratas dibuka (BatchNormalization tetap beku), Adam dengan *cosine decay* dari 1e-4, *early stopping* pada *macro-F1* validasi.

Model terbaik disimpan ke `A_PERSIAPAN/models/cnn_effb0_retrain.keras`. Bila berkas ini sudah ada, pelatihan dilewati.

In [ ]:
tf = dk.models.get_tf()
import keras
CNN_FINAL = os.path.join(P["A_model"], "cnn_effb0_retrain.keras")
CNN_HIST = os.path.join(P["A_model"], "cnn_effb0_history.csv")
AUTOTUNE = tf.data.AUTOTUNE

def sampel_roi(df, maks, seed, kolom="sample_id"):
    d = df.sample(frac=1.0, random_state=seed)
    return d[d.groupby(kolom).cumcount() < maks].reset_index(drop=True)

tr = sampel_roi(rois[rois.split == "train"], CNN_MAX_ROI_PER_IMG, SEED)
va = sampel_roi(rois[rois.split == "val"], CNN_MAX_ROI_VAL, SEED + 1)

def to_tensors(df):
    boxes = dk.geometry.boxes_to_tf_norm(df[["x1", "y1", "x2", "y2"]].to_numpy(), df["width"].to_numpy(),
                                         df["height"].to_numpy(), ROI_PAD)
    return df["path"].values.astype(str), boxes.astype(np.float32), df["sev"].values.astype(np.int32)

def load_crop(path, box, y):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.cast(img, tf.float32)[None]
    crop = tf.image.crop_and_resize(img, box[None], [0], (CNN_INPUT, CNN_INPUT))[0]
    return crop, tf.one_hot(y, 3)

rot = keras.layers.RandomRotation(10 / 360, fill_mode="reflect", seed=SEED)
zoom = keras.layers.RandomZoom(0.1, fill_mode="reflect", seed=SEED)
def augment(x, y):
    x = tf.image.random_flip_left_right(x); x = tf.image.random_flip_up_down(x)
    x = rot(x, training=True); x = zoom(x, training=True)
    b = tf.random.uniform([tf.shape(x)[0], 1, 1, 1], 0.9, 1.1)
    c = tf.random.uniform([tf.shape(x)[0], 1, 1, 1], 0.9, 1.1)
    mean = tf.reduce_mean(x, axis=[1, 2, 3], keepdims=True)
    x = (x * b - mean) * c + mean
    x = x + tf.random.normal(tf.shape(x), 0.0, 3.0)
    return tf.clip_by_value(x, 0.0, 255.0), y

def make_ds(df, training):
    ds = tf.data.Dataset.from_tensor_slices(to_tensors(df))
    if training:
        ds = ds.shuffle(min(len(df), 20000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_crop, num_parallel_calls=AUTOTUNE).batch(CNN_BATCH)
    if training:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

counts = np.bincount(tr["sev"], minlength=3)
class_weight = {k: float(len(tr) / (3 * max(counts[k], 1))) for k in range(3)}
print("ROI train:", len(tr), "per kelas:", counts.tolist(), "| ROI val:", len(va))
print("Bobot kelas:", {NAMA_KELAS_ID[k]: round(v, 3) for k, v in class_weight.items()})

In [ ]:
if os.path.exists(CNN_FINAL):
    print("Model CNN hasil latih ulang sudah ada; pelatihan dilewati.")
    cnn_hist = pd.read_csv(CNN_HIST) if os.path.exists(CNN_HIST) else pd.DataFrame()
else:
    if tf.config.list_physical_devices("GPU"):
        keras.mixed_precision.set_global_policy("mixed_float16")
    ds_tr, ds_va = make_ds(tr, True), make_ds(va, False)
    model = dk.models.build_effnet_b0(3, CNN_INPUT, dropout=0.3)
    for l in model.layers[:-3]:
        l.trainable = False
    f1m = lambda: keras.metrics.F1Score(average="macro", name="f1_score")
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy", f1m()])
    h1 = model.fit(ds_tr, validation_data=ds_va, epochs=CNN_EPOCHS_HEAD, class_weight=class_weight, verbose=2)
    for l in model.layers[-(CNN_FT_LAYERS + 3):]:
        if not isinstance(l, keras.layers.BatchNormalization):
            l.trainable = True
    steps = max(1, int(np.ceil(len(tr) / CNN_BATCH))) * CNN_EPOCHS_FT
    model.compile(optimizer=keras.optimizers.Adam(keras.optimizers.schedules.CosineDecay(1e-4, steps)),
                  loss="categorical_crossentropy", metrics=["accuracy", f1m()])
    cbs = [keras.callbacks.ModelCheckpoint(CNN_FINAL, monitor="val_f1_score", mode="max", save_best_only=True),
           keras.callbacks.EarlyStopping(monitor="val_f1_score", mode="max", patience=6, restore_best_weights=True)]
    h2 = model.fit(ds_tr, validation_data=ds_va, epochs=CNN_EPOCHS_FT, class_weight=class_weight, callbacks=cbs, verbose=2)
    cnn_hist = pd.concat([pd.DataFrame(h1.history).assign(tahap="head"), pd.DataFrame(h2.history).assign(tahap="fine_tune")],
                         ignore_index=True)
    cnn_hist["epoch"] = np.arange(1, len(cnn_hist) + 1)
    cnn_hist.to_csv(CNN_HIST, index=False)
    if not os.path.exists(CNN_FINAL):
        model.save(CNN_FINAL)
    keras.mixed_precision.set_global_policy("float32")
if len(cnn_hist):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    ax[0].plot(cnn_hist.epoch, cnn_hist.loss, label="train"); ax[0].plot(cnn_hist.epoch, cnn_hist.val_loss, label="val")
    ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss"); ax[0].legend()
    ax[1].plot(cnn_hist.epoch, cnn_hist.f1_score, label="train macro-F1"); ax[1].plot(cnn_hist.epoch, cnn_hist.val_f1_score, label="val macro-F1")
    ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Macro-F1"); ax[1].legend()
    nh = int((cnn_hist.tahap == "head").sum())
    for a in ax:
        a.axvline(nh + 0.5, color="gray", ls="--", lw=1)
    dk.viz.show_and_save(fig, os.path.join(P["A_fig"], "A4_cnn_kurva_latih"))
    b = cnn_hist.loc[cnn_hist.val_f1_score.idxmax()]
    print(f"Interpretasi: macro-F1 validasi tertinggi {dk.teks.fmt(b.val_f1_score)} pada epoch {int(b.epoch)} (tahap {b.tahap}); "
          f"macro-F1 train pada epoch yang sama {dk.teks.fmt(b.f1_score)}. Selisih train-val = {dk.teks.fmt(b.f1_score - b.val_f1_score)}.")

### A4.4 Memuat dan memverifikasi bobot lama

Bobot lama dicari otomatis di Google Drive dengan nama `model-yolo.pt` dan `best_cnn_model.h5` (kedalaman maksimal 6 folder, batas 4 menit), atau diambil dari `BOBOT_LAMA_YOLO`/`BOBOT_LAMA_CNN` bila diisi. Verifikasi yang dilakukan:

1. **YOLO lama**: jenis tugas, jumlah dan nama kelas. Semua kelas dihitung sebagai lesi kecuali `OLD_YOLO_KELAS_LESI` diisi.
2. **CNN lama**: ukuran masukan (catatan proyek menyebut 128×128 saat pelatihan dan 224×224 pada jalur lain), jumlah keluaran, skala piksel, dan urutan kelas. Untuk skala dan urutan bernilai `"auto"`, kita menguji tiga skala (0–1, 0–255, −1–1) pada ROI referensi *split val*, mencari permutasi kelas yang memaksimalkan diagonal *confusion matrix* (Hungarian), lalu memilih kombinasi dengan *macro-F1* tertinggi. Pilihan ini perlu dicocokkan dengan notebook pelatihan lama (urutan folder kelas).
3. **Penanganan YOLO tanpa deteksi**: sama untuk kedua model, yaitu lesi tanpa ROI valid dihitung salah pada metrik *end-to-end*.

Karena seluruh citra repo kemungkinan besar sudah dilihat bobot lama, semua hasil bobot lama diberi label *in-sample*.

In [ ]:
from scipy.optimize import linear_sum_assignment
old_yolo_path, old_cnn_path = BOBOT_LAMA_YOLO, BOBOT_LAMA_CNN
if CARI_BOBOT_LAMA and (old_yolo_path is None or old_cnn_path is None):
    print("Mencari bobot lama di Drive ...")
    found = dk.models.find_files(DRIVE_BASE, list(NAMA_BERKAS_BOBOT_LAMA), max_depth=6, timeout=240)
    for n, lst in found.items():
        print(f"  {n}: {lst if lst else 'tidak ditemukan'}")
    old_yolo_path = old_yolo_path or (found[NAMA_BERKAS_BOBOT_LAMA[0]][0] if found[NAMA_BERKAS_BOBOT_LAMA[0]] else None)
    old_cnn_path = old_cnn_path or (found[NAMA_BERKAS_BOBOT_LAMA[1]][0] if found[NAMA_BERKAS_BOBOT_LAMA[1]] else None)

old_info = {"yolo_path": old_yolo_path, "cnn_path": old_cnn_path, "tersedia": False}
if old_yolo_path and old_cnn_path and os.path.exists(old_yolo_path) and os.path.exists(old_cnn_path):
    oy = dk.models.YoloRunner(old_yolo_path, imgsz=YOLO_IMGSZ, nms_iou=NMS_IOU, max_det=MAX_DET,
                              keep_classes=OLD_YOLO_KELAS_LESI, name="lama")
    old_info.update({"yolo_task": getattr(oy.model, "task", None), "yolo_names": oy.names,
                     "yolo_sha256": dk.core.sha256_file(old_yolo_path)})
    ocnn, backend = dk.models.load_keras_any(old_cnn_path)
    shp = ocnn.input_shape[0] if isinstance(ocnn.input_shape, list) else ocnn.input_shape
    old_info.update({"cnn_backend": backend, "cnn_input_shape": list(shp), "cnn_n_out": int(ocnn.output_shape[-1]),
                     "cnn_sha256": dk.core.sha256_file(old_cnn_path)})
    nm_y = old_info["yolo_names"]
    print("YOLO lama:", old_info["yolo_task"], "| jumlah kelas:", len(nm_y), "|", dict(list(nm_y.items())[:10]), "..." if len(nm_y) > 10 else "")
    print("CNN lama : input", old_info["cnn_input_shape"], "| keluaran", old_info["cnn_n_out"], "| backend", backend)

    va_chk = sampel_roi(rois[rois.split == "val"], 3, SEED + 7)
    va_chk = sampel_roi(va_chk, 300, SEED, kolom="sev")
    crops, ys = [], []
    for sid, g in va_chk.groupby("sample_id"):
        img = dk.data.read_rgb(g.path.iloc[0])
        crops.append(dk.models.crop_rois(img, g[["x1", "y1", "x2", "y2"]].to_numpy(), OLD_ROI_PAD, int(shp[1])))
        ys.append(g.sev.values)
    crops, ys = np.concatenate(crops), np.concatenate(ys)
    skala_uji = ["0-1", "0-255", "-1-1"] if OLD_CNN_SKALA == "auto" else [OLD_CNN_SKALA]
    hasil_cek = []
    for sk in skala_uji:
        raw = dk.models.CnnRunner(ocnn, scale=sk).predict_raw(crops)
        cm = np.zeros((raw.shape[1], 3))
        for p_, t_ in zip(raw.argmax(1), ys):
            cm[p_, t_] += 1
        if OLD_CNN_URUTAN_KELAS == "auto":
            r_, c_ = linear_sum_assignment(-cm)
            order = [-1] * raw.shape[1]
            for a, b in zip(r_, c_):
                order[a] = int(b)
        else:
            order = list(OLD_CNN_URUTAN_KELAS)
        pr = dk.models.CnnRunner(ocnn, scale=sk, class_order=order).predict(crops)
        mm = dk.metrics.cls_metrics(ys, pr.argmax(1))
        hasil_cek.append({"skala": sk, "urutan_kelas": order, "macro_f1_val": mm["macro_f1"], "accuracy_val": mm["accuracy"],
                          "prob_maks_rerata": float(raw.max(1).mean())})
    cek_df = pd.DataFrame(hasil_cek).sort_values("macro_f1_val", ascending=False)
    display(cek_df)
    pilih = cek_df.iloc[0]
    old_info.update({"tersedia": True, "cnn_scale": pilih.skala, "cnn_class_order": list(pilih.urutan_kelas),
                     "cnn_pad": OLD_ROI_PAD, "verifikasi": cek_df.to_dict(orient="records"),
                     "status_data": "IN-SAMPLE terhadap seluruh citra repo"})
    print(f"Interpretasi: kombinasi terbaik skala {pilih.skala} dengan urutan keluaran -> kelas {list(pilih.urutan_kelas)} "
          f"(macro-F1 val {dk.teks.fmt(pilih.macro_f1_val)}). "
          + ("Selisih antar-skala besar, sehingga skala piksel model lama dapat ditentukan dari data. " if cek_df.macro_f1_val.max() - cek_df.macro_f1_val.min() > 0.1 else
             "Selisih antar-skala kecil, sehingga skala piksel belum dapat dipastikan dari data; cocokkan dengan notebook pelatihan lama. ")
          + "Angka ini in-sample dan hanya dipakai untuk verifikasi preprocessing.")
else:
    print("Bobot lama tidak ditemukan. Analisis berjalan dengan model latih ulang saja.")

### A4.5 `model_config.json`

Berkas ini mencatat semua yang diperlukan untuk mereproduksi inferensi: lokasi dan SHA-256 bobot, ukuran masukan, skala piksel, urutan kelas, padding ROI, ambang *confidence*, NMS IoU, `max_det`, parameter pelatihan, status label map, dan status data bobot lama. Notebook 01 dan 02 memuat model hanya melalui berkas ini.

In [ ]:
model_config = {
    "retrain": {
        "yolo_path": Y_FINAL, "yolo_sha256": dk.core.sha256_file(Y_FINAL) if os.path.exists(Y_FINAL) else None,
        "yolo_train_args": {k: (v if not isinstance(v, tuple) else list(v)) for k, v in yolo_args.items()},
        "yolo_keep_classes": None,
        "cnn_path": CNN_FINAL, "cnn_sha256": dk.core.sha256_file(CNN_FINAL) if os.path.exists(CNN_FINAL) else None,
        "cnn_input": CNN_INPUT, "cnn_scale": "0-255", "cnn_class_order": [0, 1, 2], "cnn_pad": ROI_PAD,
        "cnn_train": {"arsitektur": "EfficientNet-B0 (ImageNet)", "epochs_head": CNN_EPOCHS_HEAD, "epochs_ft": CNN_EPOCHS_FT,
                      "ft_layers": CNN_FT_LAYERS, "batch": CNN_BATCH, "max_roi_per_img": CNN_MAX_ROI_PER_IMG,
                      "class_weight": class_weight,
                      "augmentasi": "flip H/V, rotasi +/-10 derajat, zoom 10%, brightness/contrast +/-10%, gaussian noise sigma 3"},
        "status_data": "split terkunci: test tidak pernah dipakai untuk pelatihan atau pemilihan model",
    },
    "lama": {**old_info, "yolo_keep_classes": OLD_YOLO_KELAS_LESI, "cnn_input": old_info.get("cnn_input_shape", [None, None])[1]
             if old_info.get("tersedia") else None},
    "inferensi": {"conf_thr": CONF_THR, "nms_iou": NMS_IOU, "iou_valid": IOU_VALID, "max_det": MAX_DET, "yolo_imgsz": YOLO_IMGSZ,
                  "conf_untuk_map": 0.001, "agnostic_nms": True},
    "label_map": {"map": {str(k): v for k, v in LABEL_MAP.items()}, "status": STATUS_LABEL_MAP},
    "split_sha256": hash_split(split_df), "repo_commit": repo_commit,
}
dk.core.save_json(model_config, os.path.join(P["A_tab"], "model_config.json"))
with open(os.path.join(P["A_tab"], "environment.txt"), "w") as f:
    f.write(dk.core.env_report())
print(json.dumps(model_config["inferensi"], indent=1))
LOG("A4 selesai: model_config.json tersimpan.")

## A5. *Baseline* pada *test set* yang sah

Satu fungsi inferensi terstandar (`dk.pipeline.run_condition`) dijalankan pada *test set* terkunci tanpa gangguan. Untuk setiap model (latih ulang sebagai analisis utama, bobot lama sebagai pembanding *in-sample*) disimpan:

- `baseline_predictions_roi_<model>.csv`: satu baris per lesi referensi: kelas benar, probabilitas CNN pada ROI referensi, pasangan kotak YOLO, IoU, *confidence*, probabilitas CNN pada ROI prediksi, penanda ROI valid, dan prediksi *pipeline* (−1 = gagal);
- `baseline_predictions_image_<model>.csv`: satu baris per citra: jumlah lesi, jumlah deteksi, label tingkat citra, latensi YOLO dan CNN;
- `baseline_det_<model>.npz`: penanda TP dan *confidence* untuk mAP.

Metrik dihitung dengan 95% CI *cluster bootstrap* (`B_BOOT` replikasi):

| Komponen | Metrik |
|---|---|
| YOLO | *precision* dan *recall* lesi pada `CONF_THR` (Hungarian, IoU ≥ 0,5), mAP@0.5, mAP@0.5:0.95 |
| CNN (ROI referensi, E1) | *accuracy*, *macro-F1*, *balanced accuracy* |
| *Pipeline* YOLO–CNN (E2) | *accuracy*, *macro-F1*, *balanced accuracy*, *detection failure rate* (lesi tanpa ROI valid) |
| Tingkat citra | *macro-F1* label keparahan maksimum, *failure rate* citra (citra berlesi tanpa deteksi) |

Sebagai pemeriksaan silang, mAP model latih ulang juga dihitung dengan `model.val()` Ultralytics.

In [ ]:
items_test = dk.pipeline.build_items(split_df, ver, REPO_DIR, "test")
print("Citra test:", len(items_test), "| lesi test:", sum(len(i["gt_cls"]) for i in items_test))
cnn_retrain = keras.models.load_model(CNN_FINAL, compile=False)
MODEL_SETS = {"retrain": (dk.models.YoloRunner(Y_FINAL, imgsz=YOLO_IMGSZ, nms_iou=NMS_IOU, max_det=MAX_DET, name="retrain"),
                          dk.models.CnnRunner(cnn_retrain, scale="0-255", class_order=[0, 1, 2], pad=ROI_PAD, name="retrain"))}
if old_info.get("tersedia"):
    MODEL_SETS["lama"] = (oy, dk.models.CnnRunner(ocnn, scale=old_info["cnn_scale"], class_order=old_info["cnn_class_order"],
                                                  pad=OLD_ROI_PAD, name="lama"))
cond0 = dk.perturb.list_conditions()[0]
BASE = {}
for nm, (yr, cr) in MODEL_SETS.items():
    t0 = time.time()
    roi_df, img_df, det = dk.pipeline.run_condition(items_test, yr, cr, cond0, CONF_THR, IOU_VALID, desc=f"baseline {nm}")
    roi_df.to_csv(os.path.join(P["A_tab"], f"baseline_predictions_roi_{nm}.csv"), index=False)
    img_df.to_csv(os.path.join(P["A_tab"], f"baseline_predictions_image_{nm}.csv"), index=False)
    dk.pipeline.save_det(det, os.path.join(P["A_tab"], f"baseline_det_{nm}.npz"))
    BASE[nm] = (roi_df, img_df, det)
    print(f"{nm}: {len(roi_df):,} ROI, {len(img_df):,} citra, {time.time() - t0:.0f} detik")

In [ ]:
def metrik_baseline(roi, img, det, ri=None, ii=None, sids=None):
    r = roi if ri is None else roi.iloc[ri]
    im = img if ii is None else img.iloc[ii]
    ap = dk.metrics.ap_from_det(det, sids if sids is not None else list(det.keys()), CONF_THR)
    n_pred = im["n_pred"].sum()
    out = {"yolo_precision": r["valid_roi"].sum() / n_pred if n_pred else np.nan,
           "yolo_recall": r["valid_roi"].mean(), "yolo_map50": ap["map50"], "yolo_map50_95": ap["map50_95"]}
    for pre, col in [("cnn_ref", "ref_pred"), ("pipeline", "pipe_pred")]:
        m = dk.metrics.cls_metrics(r["true"], r[col])
        out.update({f"{pre}_accuracy": m["accuracy"], f"{pre}_macro_f1": m["macro_f1"],
                    f"{pre}_balanced_accuracy": m["balanced_accuracy"]})
    out["pipeline_detection_failure_rate"] = float((r["pipe_pred"] == -1).mean())
    il = im[im["true_img"] >= 0]
    out["image_pipeline_macro_f1"] = dk.metrics.macro_f1_fast(il["true_img"], il["pipe_img"])
    out["image_failure_rate"] = float((~il["any_det"].astype(bool)).mean()) if len(il) else np.nan
    return out

METRIK = {}
for nm, (roi_df, img_df, det) in BASE.items():
    est = metrik_baseline(roi_df, img_df, det)
    sid_list = list(det.keys())
    g_of = dict(zip(img_df.sample_id, img_df.group))
    cb = dk.stats.ClusterBootstrap(B_BOOT, SEED).register("roi", roi_df.group.values).register("img", img_df.group.values)
    cb.register("det", np.array([g_of[s] for s in sid_list]))
    boots = [metrik_baseline(roi_df, img_df, det, d["roi"], d["img"], [sid_list[i] for i in d["det"]]) for d in cb.draws()]
    bdf = pd.DataFrame(boots)
    METRIK[nm] = {k: {"estimate": v, "ci_lo": dk.stats.percentile_ci(bdf[k])[0], "ci_hi": dk.stats.percentile_ci(bdf[k])[1]}
                  for k, v in est.items()}
    METRIK[nm]["n"] = {"citra": len(img_df), "lesi": len(roi_df), "kelompok": int(img_df.group.nunique())}
    METRIK[nm]["latensi_ms"] = {"yolo_median": float(img_df.yolo_ms.median()), "cnn_median": float(img_df.cnn_ms.median()),
                                "total_median": float((img_df.yolo_ms + img_df.cnn_ms).median())}

try:
    v = YOLO(Y_FINAL).val(data=YOLO_YAML, split="test", imgsz=YOLO_IMGSZ, batch=YOLO_BATCH, conf=0.001, iou=NMS_IOU,
                          single_cls=True, plots=False, verbose=False, project=os.path.join(WORK, "val_tmp"), name="test", exist_ok=True)
    METRIK["retrain"]["ultralytics_val_test"] = {"map50": float(v.box.map50), "map50_95": float(v.box.map),
                                                 "precision": float(v.box.mp), "recall": float(v.box.mr)}
except Exception as e:
    METRIK["retrain"]["ultralytics_val_test"] = {"error": str(e)[:200]}
METRIK["catatan"] = {"label_map_status": STATUS_LABEL_MAP, "lama": "in-sample", "B_boot": B_BOOT,
                     "unit_bootstrap": "kelompok anti-leakage (sumber/sesi/near-duplicate)"}
dk.core.save_json(METRIK, os.path.join(P["A_tab"], "metrics.json"))

baris = [("YOLO", "Precision (CONF_THR)", "yolo_precision"), ("YOLO", "Recall (CONF_THR)", "yolo_recall"),
         ("YOLO", "mAP@0.5", "yolo_map50"), ("YOLO", "mAP@0.5:0.95", "yolo_map50_95"),
         ("CNN (ROI referensi)", "Accuracy", "cnn_ref_accuracy"), ("CNN (ROI referensi)", "Macro-F1", "cnn_ref_macro_f1"),
         ("CNN (ROI referensi)", "Balanced accuracy", "cnn_ref_balanced_accuracy"),
         ("Pipeline YOLO-CNN", "Accuracy", "pipeline_accuracy"), ("Pipeline YOLO-CNN", "Macro-F1", "pipeline_macro_f1"),
         ("Pipeline YOLO-CNN", "Balanced accuracy", "pipeline_balanced_accuracy"),
         ("Pipeline YOLO-CNN", "Detection failure rate", "pipeline_detection_failure_rate"),
         ("Tingkat citra", "Macro-F1 (keparahan maks)", "image_pipeline_macro_f1"), ("Tingkat citra", "Failure rate citra", "image_failure_rate")]
tab5 = []
for nm in BASE:
    for komp, met, key in baris:
        e = METRIK[nm][key]
        tab5.append({"model": nm, "Komponen": komp, "Metrik": met, "Nilai": round(e["estimate"], 4),
                     "95% CI": f"[{e['ci_lo']:.4f}; {e['ci_hi']:.4f}]"})
tab5 = pd.DataFrame(tab5)
tab5.to_csv(os.path.join(P["A_tab"], "tabel5_baseline.csv"), index=False)
display(tab5[tab5.model == "retrain"])
print("Pemeriksaan silang Ultralytics (test):", METRIK["retrain"].get("ultralytics_val_test"))

### A5.1 *Confusion matrix* ROI referensi dan ROI prediksi (Gambar 5)

Kolom terakhir pada matriks ROI prediksi adalah lesi tanpa ROI valid (tidak terdeteksi atau IoU < 0,5). Nilai dinormalisasi per baris (proporsi terhadap jumlah lesi kelas sebenarnya), dengan jumlah absolut di dalam kurung.

In [ ]:
roi_r = BASE["retrain"][0]
fig, ax = plt.subplots(1, 2, figsize=(14, 5.6))
for a, (col, judul, ext) in zip(ax, [("ref_pred", "Reference ROI (E1)", False), ("pipe_pred", "YOLO-predicted ROI (E2)", True)]):
    cm = dk.metrics.confusion(roi_r["true"], roi_r[col])
    cm = cm if ext else cm[:, :3]
    norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    a.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    xl = NAMA_KELAS + (["No valid ROI"] if ext else [])
    a.set_xticks(range(len(xl))); a.set_xticklabels(xl, rotation=20)
    a.set_yticks(range(3)); a.set_yticklabels(NAMA_KELAS)
    a.set_xlabel("Predicted"); a.set_ylabel("True"); a.set_title(judul); a.grid(False)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            a.text(j, i, f"{norm[i, j]:.2f}\n({cm[i, j]:,})", ha="center", va="center", fontsize=12,
                   color="white" if norm[i, j] > 0.55 else "black")
dk.viz.show_and_save(fig, os.path.join(P["A_fig"], "Gambar5_confusion_matrix_clean"))

### A5.2 Interpretasi otomatis *baseline*

In [ ]:
fmt, fci, pct = dk.teks.fmt, dk.teks.fmt_ci, dk.teks.pct
M = METRIK["retrain"]
g = lambda k: (M[k]["estimate"], M[k]["ci_lo"], M[k]["ci_hi"])
teks = []
teks.append(f"Test set terkunci berisi {M['n']['citra']:,} citra, {M['n']['lesi']:,} lesi, dan {M['n']['kelompok']} kelompok independen. "
            "Interval kepercayaan dihitung per kelompok.")
e = g("yolo_map50"); e2 = g("yolo_map50_95")
teks.append(f"YOLO latih ulang mencapai mAP@0.5 {fmt(e[0])} (95% CI {fci(e[1], e[2])}) dan mAP@0.5:0.95 {fmt(e2[0])} (95% CI {fci(e2[1], e2[2])}). "
            f"Pada ambang operasi {fmt(CONF_THR, 2)}, precision lesi {fmt(g('yolo_precision')[0])} dan recall lesi {fmt(g('yolo_recall')[0])}.")
uv = M.get("ultralytics_val_test", {})
if "map50" in uv:
    teks.append(f"Pemeriksaan silang Ultralytics: mAP@0.5 {fmt(uv['map50'])}, mAP@0.5:0.95 {fmt(uv['map50_95'])} "
                f"(selisih terhadap perhitungan kita {fmt(abs(uv['map50'] - e[0]), 4)} dan {fmt(abs(uv['map50_95'] - e2[0]), 4)}).")
a, b = g("cnn_ref_macro_f1"), g("pipeline_macro_f1")
teks.append(f"Macro-F1 CNN pada ROI referensi (E1) = {fmt(a[0])} (95% CI {fci(a[1], a[2])}); macro-F1 pipeline dengan ROI prediksi (E2) = "
            f"{fmt(b[0])} (95% CI {fci(b[1], b[2])}). Selisih E1 - E2 = {fmt(a[0] - b[0])}, indikator awal penalti lokalisasi; "
            "uji berpasangan lengkap (Delta L) ada di notebook 01.")
f = g("pipeline_detection_failure_rate")
teks.append(f"Detection failure rate (lesi tanpa ROI valid) = {pct(f[0])} (95% CI {pct(f[1])} - {pct(f[2])}). Lesi ini tetap dihitung salah pada metrik end-to-end.")
cmr = dk.metrics.confusion(roi_r["true"], roi_r["ref_pred"])[:, :3]
off = [(i, j, cmr[i, j]) for i in range(3) for j in range(3) if i != j]
i, j, n = max(off, key=lambda t: t[2])
teks.append(f"Kesalahan terbanyak pada ROI referensi adalah {NAMA_KELAS_ID[i]} diprediksi {NAMA_KELAS_ID[j]} ({n:,} lesi, "
            f"{pct(n / max(cmr[i].sum(), 1))} dari lesi {NAMA_KELAS_ID[i]}).")
teks.append(f"Median latensi per citra: YOLO {fmt(M['latensi_ms']['yolo_median'], 1)} ms, CNN {fmt(M['latensi_ms']['cnn_median'], 1)} ms.")
if "lama" in METRIK:
    L = METRIK["lama"]
    teks.append(f"Bobot lama (in-sample, bukan unseen test): mAP@0.5 {fmt(L['yolo_map50']['estimate'])}, macro-F1 E1 "
                f"{fmt(L['cnn_ref_macro_f1']['estimate'])}, macro-F1 E2 {fmt(L['pipeline_macro_f1']['estimate'])}. "
                "Angka ini tidak dibandingkan sebagai kinerja generalisasi.")
teks.append(f"Status label map: {STATUS_LABEL_MAP}. Laporan Setiawan et al. (2026) dipakai hanya sebagai konteks karena data uji, "
            "preprocessing, dan standar referensinya berbeda.")
print("\n\n".join(teks))
LOG("A5 selesai: baseline tersimpan.")

## Ringkasan *checklist* Tahap A dan unduhan

Sel terakhir menampilkan status *checklist* A1–A5 berdasarkan keberadaan berkas keluaran, lalu mengemas folder `A_PERSIAPAN` (tanpa bobot model dan folder *run* YOLO agar ukurannya kecil) menjadi ZIP dan mengunduhnya.

In [ ]:
cek_file = {
    "01 Audit dataset dan duplikat": ["dataset_manifest.csv", "duplicate_report.csv"],
    "02 Validasi label dan anotasi": ["verified_labels.csv", "label_mapping.json"],
    "03 Kunci split dan audit leakage": ["locked_data_split.csv", "leakage_audit.csv"],
    "04 Verifikasi loading YOLO dan CNN": ["model_config.json", "environment.txt"],
    "05 Evaluasi baseline": ["baseline_predictions_roi_retrain.csv", "metrics.json"],
}
for k, fs in cek_file.items():
    ok = all(os.path.exists(os.path.join(P["A_tab"], f)) for f in fs)
    print(("[x] " if ok else "[ ] ") + k)
print(f"\nCatatan: label map berstatus {STATUS_LABEL_MAP}; validasi label oleh ahli (checklist 02) belum lengkap sampai status ini diubah.")

zip_base = os.path.join(WORK, f"{NAMA_PROYEK}_A_PERSIAPAN")
tmp = os.path.join(WORK, "_zip_A")
if os.path.exists(tmp):
    shutil.rmtree(tmp)
shutil.copytree(P["A"], os.path.join(tmp, "A_PERSIAPAN"), ignore=shutil.ignore_patterns("models", "*.pt", "*.keras"))
zip_path = shutil.make_archive(zip_base, "zip", tmp)
shutil.copy2(zip_path, ROOT)
print("ZIP:", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB), salinan di", ROOT)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)